In [1]:
from VSP_Auxiliary import *
import VSP_ZTF
import VSP_ASASSN
import VSP_TESS
import matplotlib.pyplot as plt
from astropy.table import Table
from astropy.stats import sigma_clipping
from astropy.timeseries import aggregate_downsample
from astropy.timeseries import LombScargle
from astropy.timeseries import TimeSeries
from astropy.time import Time
from scipy import optimize
import lightkurve as lk
import scipy.signal
import pandas as pd
import time
print('VSP_Period Version: 20240728')

Welcome to ASAS-SN Skypatrol!

Current Deployment Version: 0.6.17 (26 JAN 2024)
Please upgrade your client if not up to date.



In [2]:
def get_lc(coord, path, name='', select=['ztf','asassn','tess']):
    identifier = coord2name(coord)
    prefix = name+'_'+identifier
    
    if 'ztf' in select:
        print('>>> Getting ZTF data....')
        lc_name = prefix + '_ztf_lc.csv'
        lc_path = path + lc_name
        png_name = prefix + '_ztf_lc.png'
        png_path = path + png_name
        if VSP_ZTF.download(coord, lc_path):
            # 当下载到光变曲线
            print('ZTF lc download success.')
            try:
                VSP_ZTF.plot_ztf(lc_path, png_path, ifshow=False)
                print('ZTF lc has been plotted.')
            except:
                time.sleep(2.5)
                VSP_ZTF.download(coord, lc_path)
                try:
                    VSP_ZTF.plot_ztf(lc_path, png_path, ifshow=False)
                    print('ZTF lc has been plotted.')
                except:
                    print('Could not read ZTF lc file.')
        else:
            # 当没有下载到光变曲线
            print('Failed downloading ZTF lc. Retrying...')
            time.sleep(10)
            if VSP_ZTF.download(coord, lc_path): # 重试下载
                print('ZTF lc downloaded.')
                try:
                    VSP_ZTF.plot_ztf(lc_path, png_path, ifshow=False)
                    print('ZTF lc has been plotted.')
                except:
                    print('Could not read ZTF lc file.')
            else:
                print('Could not download ZTF lc.')

    if 'asassn' in select:
        print('>>> Getting ASASSN data....')
        asas_pass = True
        asaslc_name = prefix + '_asassn_lc.csv'
        asaspng_name = prefix +'_asassn_lc.png'
        try:
            asas_lc = VSP_ASASSN.download(coord, path, asaslc_name)
        except:
            asas_pass = False
            print('Failed getting ASASSN data.')
        if asas_pass and asas_lc:
            VSP_ASASSN.plot_lc(path+asaslc_name, save_path=path+asaspng_name, ifshow=False)
            print('ASASSN lc has been plotted.')
    
    if 'tess' in select:
        print('>>> Getting Kepler/TESS data....')
        VSP_TESS.get_lc(coord, lc_path=path, pic_path=path,prefix=prefix)

# 测试
#c = SkyCoord('06 52 37.19 +24 36 22.1', unit=(u.hourangle, u.deg), frame='fk5')
#lc_save_path = './Temp/lc_download_test/'
#create_dir(lc_save_path)
#get_lc(c,path=lc_save_path,name='TEST')

In [3]:
# T CrB 239.87567 25.92017
#c = SkyCoord('239.87567 +25.92017', unit=(u.deg, u.deg), frame='fk5')
#lc_save_path = './Temp/lc_download_test/'
#create_dir(lc_save_path)
#get_lc(c,path=lc_save_path,name='T CrB')

In [4]:
def cal_magerr(F,Ferr):
    return np.abs(-2.5/np.log(10)*Ferr/F)

def process_gaia_epoch(csvfile):
    data_table = Table.read(csvfile,format='ascii.csv')
    #print(data_table)
    Gtime = data_table["TimeG"] #  Transit averaged G band observing time JD-2455197.5
    GtimeJD = Gtime + 2455197.5
    Gmag = data_table["Gmag"] # Transit averaged G band Vega magnitude
    FG = data_table["FG"]
    FGerr = data_table["e_FG"]
    
    BPtime = data_table["TimeBP"] #  BP CCD transit observing time JD-2455197.5
    BPtimeJD = BPtime + 2455197.5
    BPmag = data_table["BPmag"] # BP band Vega magnitude
    FBP = data_table["FBP"]
    FBPerr = data_table["e_FBP"]
    
    RPtime = data_table["TimeRP"] #  Transit averaged G band observing time JD-2455197.5
    RPtimeJD = RPtime + 2455197.5
    RPmag = data_table["RPmag"] # RP CCD transit observing time JD-2455197.5 
    FRP = data_table["FRP"]
    FRPerr = data_table["e_FRP"]
    
    Gmagerr = cal_magerr(FG,FGerr)
    BPmagerr = cal_magerr(FBP,FBPerr)
    RPmagerr = cal_magerr(FRP,FRPerr)
    #return (GtimeJD,Gmag,Gmagerr,BPtimeJD,BPmag,BPmagerr,RPtimeJD,RPmag,RPmagerr)
    return (GtimeJD.value,Gmag.value,Gmagerr.value,BPtimeJD.value,BPmag.value,BPmagerr.value,RPtimeJD.value,RPmag.value,RPmagerr.value)

#file = './Temp/DR3_epoch.csv'
#process_gaia_epoch(file)

In [5]:
def plot_ztf_asassn(ztf_lc=None,asassn_lc=None,gaia_lc=None,save_path=None,ifshow=False,dpi=360):

    asassn_valid = True
    ztf_valid = True
    gaia_valid = True
    if ztf_lc is None:
        ztf_valid = False
    if asassn_lc is None:
        asassn_valid = False
    if gaia_lc is None:
        gaia_valid = False
        
    if ztf_valid:
        # 读入ZTF数据
        tb1,tb2,tb3 = VSP_ZTF.read_lc(ztf_lc)
        zg_hjd1,zg_mag1,zg_err1 = tb1['hjd_g'].value,tb1['mag_g'].value,tb1['magerr_g'].value
        zr_hjd1,zr_mag1,zr_err1 = tb2['hjd_r'].value,tb2['mag_r'].value,tb2['magerr_r'].value
        zi_hjd1,zi_mag1,zi_err1 = tb3['hjd_i'].value,tb3['mag_i'].value,tb3['magerr_i'].value
        for ee in range(len(zg_err1)):
            if zg_err1[ee]<0:
                zg_err1[ee]=np.nan
        for ee in range(len(zr_err1)):
            if zr_err1[ee]<0:
                zr_err1[ee]=np.nan
        for ee in range(len(zi_err1)):
            if zi_err1[ee]<0:
                zi_err1[ee]=np.nan
        # 计算ztf观测范围
        if len(zg_hjd1)!=0 or len(zr_hjd1)!=0 or len(zi_hjd1)!=0: # 不是都为0
            min_jd,max_jd,min_mag,max_mag = VSP_ZTF.cal_range(zg_hjd1,zr_hjd1,zi_hjd1,zg_mag1,zr_mag1,zi_mag1)
            ztf_jd_range = (min_jd,max_jd)
            ztf_mag_range = (min_mag,max_mag)
        else:
            ztf_valid = False
    if asassn_valid:
        # 读入ASASSN数据
        dataV, datag, header = VSP_ASASSN.output_data(asassn_lc)
        # 计算asassn观测范围
        if len(dataV)==0 and len(datag)>0:
            max_jd = max(datag['jd'])
            min_jd = min(datag['jd'])
        elif len(datag)==0 and len(dataV)>0:
            max_jd = max(dataV['jd'])
            min_jd = min(dataV['jd'])
        elif len(dataV)>0 and len(datag)>0:
            max_jd = max(max(dataV['jd']),max(datag['jd']))
            min_jd = min(min(dataV['jd']),min(datag['jd']))
        else:
            asassn_valid = False  
    if asassn_valid:
        asas_jd_range = (min_jd,max_jd)
        try:
            asas_minmag = min(min(dataV['mag']),min(datag['mag']))
            asas_maxmag = max(max(dataV['mag']),max(datag['mag']))
        except:
            try:
                asas_minmag = min(dataV['mag'])
                asas_maxmag = max(dataV['mag'])
            except:
                try:
                    asas_minmag = min(datag['mag'])
                    asas_maxmag = max(datag['mag'])
                except:
                    asassn_valid = False                
        
    if gaia_valid:
        # 读入gaia数据
        GtimeJD,Gmag,Gmagerr,BPtimeJD,BPmag,BPmagerr,RPtimeJD,RPmag,RPmagerr=process_gaia_epoch(gaia_lc)
        if len(GtimeJD)<1:
            gaia_valid = False

    final_valid = True
    if asassn_valid and ztf_valid: # 全都有
        minmag = min(asas_minmag,ztf_mag_range[0])
        maxmag = max(asas_maxmag,ztf_mag_range[1])
    elif asassn_valid: # 只有asassn有
        minmag = asas_minmag
        maxmag = asas_maxmag      
    elif ztf_valid:
        minmag = ztf_mag_range[0]
        maxmag = ztf_mag_range[1]       
    else:
        final_valid = False
        
    if final_valid:
        #print(maxmag,minmag)
        if maxmag < 10.7: # 最亮星4-11   
            magrange = np.arange(11, 4-0.5, -0.5) 
        elif maxmag < 13.7 and minmag > 7.: # 亮星7-14
            magrange = np.arange(14, 7-0.5, -0.5) 
        elif maxmag < 17.7 and minmag > 11.: # 暗星10-17
            magrange = np.arange(18, 11-0.5,-0.5) 
        elif maxmag < 20.8 and minmag > 14.: # 最暗星14-21
            magrange = np.arange(21, 14-0.5,-0.5)
        else: # 大振幅
            magrange = np.arange(21, 11-0.5, -0.5) 
            
        t0 = 56850 # 58250
        tc = 1000 # 800
        
        fs1 = 8.5
        fs2 = 7.5
        gg = 5.
        plt.figure(figsize=(18,4*3),dpi=dpi)
        plt.subplots_adjust(wspace=0, hspace=0.105) # 调整子图间距
        plt.subplot(411)
        mjd_min = t0
        mjd_max = mjd_min+tc+1
        x_ticks = np.arange(mjd_min,mjd_max,10)
        plt.xticks(x_ticks,[x_ticks[jj] if x_ticks[jj]%50==0 else '' for jj in range(len(x_ticks))],fontsize=fs2)
        plt.xlim(mjd_min,mjd_max)
        plt.yticks(magrange,[magrange[jj] if magrange[jj]%1==0 else '' for jj in range(len(magrange))], fontsize=fs2)
        plt.ylim(max(magrange),min(magrange))
        #plt.rcParams['xtick.direction'] = 'in'# 将x周的刻度线方向设置向内
        plt.rcParams['ytick.direction'] = 'in'# 将y轴的刻度方向设置向内
        plt.grid(True,zorder=0,linewidth=0.2,alpha=0.2,c='grey')
        if ztf_valid:
            ztf_zg=plt.scatter(zg_hjd1-2400000.5, zg_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:blue', alpha=0.9,zorder=4)
            ztf_zr=plt.scatter(zr_hjd1-2400000.5, zr_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:orange', alpha=0.9,zorder=4)
            ztf_zi=plt.scatter(zi_hjd1-2400000.5, zi_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:red', alpha=0.9,zorder=4)
            plt.errorbar(zg_hjd1-2400000.5, zg_mag1, yerr=zg_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:blue', alpha=0.66,zorder=3)
            plt.errorbar(zr_hjd1-2400000.5, zr_mag1, yerr=zr_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:orange', alpha=0.66,zorder=3)
            plt.errorbar(zi_hjd1-2400000.5, zi_mag1, yerr=zi_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:red', alpha=0.66,zorder=3)
        if asassn_valid:
            asasv=plt.scatter(dataV['jd']-2400000.5, dataV['mag'], 
                              marker='o', s=9, edgecolors='k', linewidths=0.4, color='tab:green', alpha=0.9,zorder=4)
            asasg=plt.scatter(datag['jd']-2400000.5, datag['mag'], 
                              marker='o', s=9, edgecolors='k', linewidths=0.4, color='tab:purple', alpha=0.9,zorder=4)
            plt.errorbar(dataV['jd']-2400000.5, dataV['mag'], yerr=dataV['mag_err'],
                         linewidth=0, elinewidth=.5, capsize=2., capthick=.5, color='tab:green', alpha=0.66,zorder=3)
            plt.errorbar(datag['jd']-2400000.5, datag['mag'], yerr=datag['mag_err'],
                         linewidth=0, elinewidth=.5, capsize=2., capthick=.5, color='tab:purple', alpha=0.66,zorder=3)
        if gaia_valid:
            dr3g=plt.scatter(GtimeJD-2400000.5, Gmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='gold', alpha=0.88,zorder=6)
            dr3bp=plt.scatter(BPtimeJD-2400000.5, BPmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='b', alpha=0.88,zorder=4)
            dr3rp=plt.scatter(RPtimeJD-2400000.5, RPmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='r', alpha=0.88,zorder=4)
            plt.errorbar(GtimeJD-2400000.5, Gmag, yerr=Gmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='gold', alpha=0.66,zorder=3)
            plt.errorbar(BPtimeJD-2400000.5, BPmag, yerr=BPmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='b', alpha=0.66,zorder=3)
            plt.errorbar(RPtimeJD-2400000.5, RPmag, yerr=RPmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='r', alpha=0.66,zorder=3)
        plt.ylabel('mag', fontsize=fs1)
        #plt.xlabel('MJD (JD-2400000.5)',fontsize=fs1)
        tt1, tt2, tt3 = '','',''
        len_V, len_g, len_zg, len_zr, len_zi = 0,0,0,0,0 
        lenG,lenBP,lenRP = 0,0,0
        tta = ''
        ttb = ''
        if asassn_valid:
            tt1 = asassn_lc.split('/')[-1]
            len_V, len_g = len(dataV), len(datag)
            tta += (tt1+'    ')
            ttb += f'asas-V/g:{len_V}/{len_g}    '
        if ztf_valid:
            tt2 = ztf_lc.split('/')[-1]
            len_zg, len_zr, len_zi = len(zg_hjd1), len(zr_hjd1), len(zi_hjd1)
            tta += (tt2+'    ')
            ttb += f'zg/zr/zi:{len_zg}/{len_zr}/{len_zi}    '
        if gaia_valid:
            tt3 = gaia_lc.split('/')[-1]
            lenG, lenBP, lenRP = len(Gmag), len(BPmag), len(RPmag)
            tta += (tt3+'    ')
            ttb += f'G/BP/RP:{lenG}/{lenBP}/{lenRP}'
        plt.title(tta + '  ' + ttb, fontsize=fs1+0.5)
        leg1, leg2 = [], []
        if ztf_valid:
            leg1.append(ztf_zg)
            leg1.append(ztf_zr)
            leg1.append(ztf_zi) 
            leg2.append('ZTF zg')
            leg2.append('ZTF zr')
            leg2.append('ZTF zi') 
        if asassn_valid:
            leg1.append(asasv)
            leg1.append(asasg) 
            leg2.append('ASASSN V')
            leg2.append('ASASSN g') 
        if gaia_valid:
            leg1.append(dr3g)
            leg1.append(dr3bp)
            leg1.append(dr3rp) 
            leg2.append('GAIA3 G')
            leg2.append('GAIA3 BP')
            leg2.append('GAIA3 RP') 
        plt.legend(leg1, leg2, loc='upper right', fontsize=fs2-2.)

        plt.subplot(412)
        mjd_min = t0+tc
        mjd_max = mjd_min+tc+1
        x_ticks = np.arange(mjd_min,mjd_max,10)
        plt.xticks(x_ticks,[x_ticks[jj] if x_ticks[jj]%50==0 else '' for jj in range(len(x_ticks))],fontsize=fs2)
        plt.xlim(mjd_min,mjd_max)
        plt.yticks(magrange,[magrange[jj] if magrange[jj]%1==0 else '' for jj in range(len(magrange))], fontsize=fs2)
        plt.ylim(max(magrange),min(magrange))
        #plt.rcParams['xtick.direction'] = 'in'# 将x周的刻度线方向设置向内
        plt.rcParams['ytick.direction'] = 'in'# 将y轴的刻度方向设置向内
        plt.grid(True,zorder=0,linewidth=0.2,alpha=0.2,c='grey')
        if ztf_valid:
            ztf_zg=plt.scatter(zg_hjd1-2400000.5, zg_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:blue', alpha=0.9,zorder=4)
            ztf_zr=plt.scatter(zr_hjd1-2400000.5, zr_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:orange', alpha=0.9,zorder=4)
            ztf_zi=plt.scatter(zi_hjd1-2400000.5, zi_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:red', alpha=0.9,zorder=4)
            plt.errorbar(zg_hjd1-2400000.5, zg_mag1, yerr=zg_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:blue', alpha=0.66,zorder=3)
            plt.errorbar(zr_hjd1-2400000.5, zr_mag1, yerr=zr_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:orange', alpha=0.66,zorder=3)
            plt.errorbar(zi_hjd1-2400000.5, zi_mag1, yerr=zi_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:red', alpha=0.66,zorder=3)
        if asassn_valid:
            asasv=plt.scatter(dataV['jd']-2400000.5, dataV['mag'], 
                              marker='o', s=9, edgecolors='k', linewidths=0.4, color='tab:green', alpha=0.9,zorder=4)
            asasg=plt.scatter(datag['jd']-2400000.5, datag['mag'], 
                              marker='o', s=9, edgecolors='k', linewidths=0.4, color='tab:purple', alpha=0.9,zorder=4)
            plt.errorbar(dataV['jd']-2400000.5, dataV['mag'], yerr=dataV['mag_err'],
                         linewidth=0, elinewidth=.5, capsize=2., capthick=.5, color='tab:green', alpha=0.66,zorder=3)
            plt.errorbar(datag['jd']-2400000.5, datag['mag'], yerr=datag['mag_err'],
                         linewidth=0, elinewidth=.5, capsize=2., capthick=.5, color='tab:purple', alpha=0.66,zorder=3)
        if gaia_valid:
            dr3g=plt.scatter(GtimeJD-2400000.5, Gmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='gold', alpha=0.88,zorder=6)
            dr3bp=plt.scatter(BPtimeJD-2400000.5, BPmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='b', alpha=0.88,zorder=4)
            dr3rp=plt.scatter(RPtimeJD-2400000.5, RPmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='r', alpha=0.88,zorder=4)
            plt.errorbar(GtimeJD-2400000.5, Gmag, yerr=Gmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='gold', alpha=0.66,zorder=3)
            plt.errorbar(BPtimeJD-2400000.5, BPmag, yerr=BPmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='b', alpha=0.66,zorder=3)
            plt.errorbar(RPtimeJD-2400000.5, RPmag, yerr=RPmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='r', alpha=0.66,zorder=3)
        plt.ylabel('mag', fontsize=fs1)
        #plt.xlabel('MJD (JD-2400000.5)',fontsize=fs1)

        plt.subplot(413)
        mjd_min = t0+tc*2
        mjd_max = mjd_min+tc+1
        x_ticks = np.arange(mjd_min,mjd_max,10)
        plt.xticks(x_ticks,[x_ticks[jj] if x_ticks[jj]%50==0 else '' for jj in range(len(x_ticks))],fontsize=fs2)
        plt.xlim(mjd_min,mjd_max)
        plt.yticks(magrange,[magrange[jj] if magrange[jj]%1==0 else '' for jj in range(len(magrange))], fontsize=fs2)
        plt.ylim(max(magrange),min(magrange))
        #plt.rcParams['xtick.direction'] = 'in'# 将x周的刻度线方向设置向内
        plt.rcParams['ytick.direction'] = 'in'# 将y轴的刻度方向设置向内
        plt.grid(True,zorder=0,linewidth=0.2,alpha=0.2,c='grey')
        if ztf_valid:
            ztf_zg=plt.scatter(zg_hjd1-2400000.5, zg_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:blue', alpha=0.9,zorder=4)
            ztf_zr=plt.scatter(zr_hjd1-2400000.5, zr_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:orange', alpha=0.9,zorder=4)
            ztf_zi=plt.scatter(zi_hjd1-2400000.5, zi_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:red', alpha=0.9,zorder=4)
            plt.errorbar(zg_hjd1-2400000.5, zg_mag1, yerr=zg_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:blue', alpha=0.66,zorder=3)
            plt.errorbar(zr_hjd1-2400000.5, zr_mag1, yerr=zr_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:orange', alpha=0.66,zorder=3)
            plt.errorbar(zi_hjd1-2400000.5, zi_mag1, yerr=zi_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:red', alpha=0.66,zorder=3)
        if asassn_valid:
            asasv=plt.scatter(dataV['jd']-2400000.5, dataV['mag'], 
                              marker='o', s=9, edgecolors='k', linewidths=0.4, color='tab:green', alpha=0.9,zorder=4)
            asasg=plt.scatter(datag['jd']-2400000.5, datag['mag'], 
                              marker='o', s=9, edgecolors='k', linewidths=0.4, color='tab:purple', alpha=0.9,zorder=4)
            plt.errorbar(dataV['jd']-2400000.5, dataV['mag'], yerr=dataV['mag_err'],
                         linewidth=0, elinewidth=.5, capsize=2., capthick=.5, color='tab:green', alpha=0.66,zorder=3)
            plt.errorbar(datag['jd']-2400000.5, datag['mag'], yerr=datag['mag_err'],
                         linewidth=0, elinewidth=.5, capsize=2., capthick=.5, color='tab:purple', alpha=0.66,zorder=3)
        if gaia_valid:
            dr3g=plt.scatter(GtimeJD-2400000.5, Gmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='gold', alpha=0.88,zorder=6)
            dr3bp=plt.scatter(BPtimeJD-2400000.5, BPmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='b', alpha=0.88,zorder=4)
            dr3rp=plt.scatter(RPtimeJD-2400000.5, RPmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='r', alpha=0.88,zorder=4)
            plt.errorbar(GtimeJD-2400000.5, Gmag, yerr=Gmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='gold', alpha=0.66,zorder=3)
            plt.errorbar(BPtimeJD-2400000.5, BPmag, yerr=BPmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='b', alpha=0.66,zorder=3)
            plt.errorbar(RPtimeJD-2400000.5, RPmag, yerr=RPmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='r', alpha=0.66,zorder=3)
        plt.ylabel('mag', fontsize=fs1)
        #plt.xlabel('MJD (JD-2400000.5)',fontsize=fs1)

        plt.subplot(414)
        mjd_min = t0+tc*3
        mjd_max = mjd_min+tc+1
        x_ticks = np.arange(mjd_min,mjd_max,10)
        plt.xticks(x_ticks,[x_ticks[jj] if x_ticks[jj]%50==0 else '' for jj in range(len(x_ticks))],fontsize=fs2)
        plt.xlim(mjd_min,mjd_max)
        plt.yticks(magrange,[magrange[jj] if magrange[jj]%1==0 else '' for jj in range(len(magrange))], fontsize=fs2)
        plt.ylim(max(magrange),min(magrange))
        #plt.rcParams['xtick.direction'] = 'in'# 将x周的刻度线方向设置向内
        plt.rcParams['ytick.direction'] = 'in'# 将y轴的刻度方向设置向内
        plt.grid(True,zorder=0,linewidth=0.2,alpha=0.2,c='grey')
        if ztf_valid:
            ztf_zg=plt.scatter(zg_hjd1-2400000.5, zg_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:blue', alpha=0.9,zorder=4)
            ztf_zr=plt.scatter(zr_hjd1-2400000.5, zr_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:orange', alpha=0.9,zorder=4)
            ztf_zi=plt.scatter(zi_hjd1-2400000.5, zi_mag1, marker='^', s=14, edgecolors='k', linewidths=0.4, color='tab:red', alpha=0.9,zorder=4)
            plt.errorbar(zg_hjd1-2400000.5, zg_mag1, yerr=zg_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:blue', alpha=0.66,zorder=3)
            plt.errorbar(zr_hjd1-2400000.5, zr_mag1, yerr=zr_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:orange', alpha=0.66,zorder=3)
            plt.errorbar(zi_hjd1-2400000.5, zi_mag1, yerr=zi_err1, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='tab:red', alpha=0.66,zorder=3)
        if asassn_valid:
            asasv=plt.scatter(dataV['jd']-2400000.5, dataV['mag'], 
                              marker='o', s=9, edgecolors='k', linewidths=0.4, color='tab:green', alpha=0.9,zorder=4)
            asasg=plt.scatter(datag['jd']-2400000.5, datag['mag'], 
                              marker='o', s=9, edgecolors='k', linewidths=0.4, color='tab:purple', alpha=0.9,zorder=4)
            plt.errorbar(dataV['jd']-2400000.5, dataV['mag'], yerr=dataV['mag_err'],
                         linewidth=0, elinewidth=.5, capsize=2., capthick=.5, color='tab:green', alpha=0.66,zorder=3)
            plt.errorbar(datag['jd']-2400000.5, datag['mag'], yerr=datag['mag_err'],
                         linewidth=0, elinewidth=.5, capsize=2., capthick=.5, color='tab:purple', alpha=0.66,zorder=3)
        if gaia_valid:
            dr3g=plt.scatter(GtimeJD-2400000.5, Gmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='gold', alpha=0.88,zorder=6)
            dr3bp=plt.scatter(BPtimeJD-2400000.5, BPmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='b', alpha=0.88,zorder=4)
            dr3rp=plt.scatter(RPtimeJD-2400000.5, RPmag, marker='s', s=gg, edgecolors='k', linewidths=0.3, color='r', alpha=0.88,zorder=4)
            plt.errorbar(GtimeJD-2400000.5, Gmag, yerr=Gmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='gold', alpha=0.66,zorder=3)
            plt.errorbar(BPtimeJD-2400000.5, BPmag, yerr=BPmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='b', alpha=0.66,zorder=3)
            plt.errorbar(RPtimeJD-2400000.5, RPmag, yerr=RPmagerr, 
                         linewidth=0, elinewidth=0.5, capsize=2., capthick=.5, color='r', alpha=0.66,zorder=3)
        plt.ylabel('mag', fontsize=fs1)
        plt.xlabel('MJD (JD-2400000.5)',fontsize=fs1)

        if save_path is not None:
            plt.savefig(save_path,dpi=dpi,bbox_inches='tight')
        if ifshow:
            plt.show()
        plt.close()
        plt.close('all')
    else:
        pass
    return final_valid

# plot_ztf_asassn(ztf_lc='./Temp/lc_download_test/TEST_J065237.19+243622.1_ztf_lc.csv',\
#                 asassn_lc='./Temp/lc_download_test/TEST_J065237.19+243622.1_asassn_lc.csv',\
#                 gaia_lc='./Temp/DR3_epoch.csv',\
#                 save_path='./Temp/plot_lc_test10.png',ifshow=True)    

In [6]:
# 读取数据
def read_all_lc_filelists(path):
    print('>>> Start reading lc files:')
    print('    read_path:',path)
    file_names = sorted(os.listdir(path)) # 读取原始文件目录中的所有文件
    
    Keplerlist = []
    identify_string1 = '_Kepler Quarter'
    identify_string2 = '.csv'
    for name in file_names:
        if identify_string1 in name and identify_string2 in name:
            Keplerlist.append(name)
    Keplerlist = sorted(Keplerlist) 
    print('>>> Kepler file number:',len(Keplerlist))
    for name in Keplerlist:
        print('   ',name) 
        
    K2list = []
    identify_string1 = '_K2SFF_'
    identify_string2 = '.csv'
    for name in file_names:
        if identify_string1 in name and identify_string2 in name:
            K2list.append(name)
    K2list = sorted(K2list) 
    print('>>> K2-SFF file number:',len(K2list))
    for name in K2list:
        print('   ',name)   
  
    tesslist = []
    identify_string1 = '_TESS Sector'
    identify_string2 = '.csv'
    for name in file_names:
        if identify_string1 in name and identify_string2 in name:
            if 'SPOC' in name: # 不分析QLP
            #if 'SPOC' in name or 'QLP' in name:
                tesslist.append(name)
    tesslist = sorted(tesslist) 
    print('>>> TESS SPOC file number:',len(tesslist))
    for name in tesslist:
        print('   ',name)
        
    asaslist = []
    identify_string1 = 'asassn_lc'
    identify_string2 = '.csv'
    for name in file_names:
        if identify_string1 in name and identify_string2 in name:
            asaslist.append(name)
    if len(asaslist)==0:
        print('>>> No ASASSN file.')
    else:
        print('>>> ASASSN file:',asaslist)

    ztflist = []
    identify_string1 = 'ztf_lc'
    identify_string2 = '.csv'
    for name in file_names:
        if identify_string1 in name and identify_string2 in name:
            ztflist.append(name)
    if len(ztflist)==0:
        print('>>> No ZTF file.')
    else:
        print('>>> ZTF file:',ztflist)
    return (tesslist, Keplerlist, K2list, asaslist, ztflist)

#path = './RUN/20240624_test1/000000_J061149.08+224932.7_LAMOST-LB1/data/lc/'
#tesslist, Keplerlist, K2list, asaslist, ztflist = read_all_lc_filelists(path)
#tesslist, Keplerlist, K2list, asaslist, ztflist

In [7]:
def read_ztf(lc_path):  
    # 输入ztf光变曲线文件（csv）
    # 返回tb1,tb2,tb3
    return VSP_ZTF.read_lc(lc_path)

def read_asas(lc_path):
    # 输入asas光变曲线文件（csv）
    # 返回光变曲线中包含的数据(dataV, datag, header)
    return VSP_ASASSN.output_data(lc_path)

In [8]:
def read_tess(lc_path):
    # 输入TESS/Kepler光变曲线文件（csv）
    # 程序筛选QUALITY==0的数据点
    # 返回光变曲线中包含的BTJD时间、BJD时间（BTJD+2457000）、流量、流量误差(btjd,bjd,flux,fluxerr)
    TIME = np.array(csvc(lc_path,0),dtype=np.float64)
    BJD = TIME+2457000
    FLUX = csvc(lc_path,1)#
    FLUX = np.array([np.nan if i=='' else i for i in FLUX],dtype=np.float64)
    FLUXERR = csvc(lc_path,2)
    FLUXERR = np.array([np.nan if i=='' else i for i in FLUXERR],dtype=np.float64)    
    QUALITY = csvc(lc_path,13)
    QUALITY = np.array([np.nan if i=='' else int(i) for i in QUALITY])

    btjd = []
    bjd = []
    flux = []
    fluxerr = []
    for i in range(len(TIME)):
        if QUALITY[i] == 0:
            btjd.append(TIME[i])
            bjd.append(BJD[i])
            flux.append(FLUX[i])
            fluxerr.append(FLUXERR[i])
    btjd=np.array(btjd,dtype=np.float64)
    bjd=np.array(bjd,dtype=np.float64)
    flux=np.array(flux,dtype=np.float64)
    fluxerr=np.array(fluxerr,dtype=np.float64)
    print('Time Coverage:',round(max(TIME)-min(TIME),2),'d')
    print('Valid Data:',len(bjd))
    
    return (btjd,bjd,flux,fluxerr)

In [9]:
def gen_tess_lc(datatess, sigma=50, ifplot=False):
    # 输入read_tess(lc_path)返回的(btjd,bjd,flux,fluxerr)数据
    # 返回以BTJD和BJD为时间的光变曲线(lc_btjd, lc_jd)
    flux = datatess[2]
    flux_err = datatess[3]
    btjd = Time(datatess[0], format='btjd', scale='tdb')
    lc_btjd = lk.LightCurve(time = btjd, flux = flux, flux_err = flux_err).remove_outliers(sigma=sigma) 
    if ifplot:
        lc_btjd.plot()
    jds = Time(datatess[1], format='jd', scale='tdb')
    lc_jd = lk.LightCurve(time = jds, flux = flux, flux_err = flux_err).remove_outliers(sigma=sigma)
    if ifplot:
        lc_jd.plot()
    return (lc_btjd, lc_jd)

def gen_ztf_lc(dataztf, sigma=150):
    # 输入读取的ztf数据
    # 返回lightkurve光变曲线(lczg, lczr, lczi)
    
    # return (tb1,tb2,tb3)
    tb1,tb2,tb3 = dataztf
    zg_hjd1,zg_mag1,zg_err1 = tb1['hjd_g'],tb1['mag_g'],tb1['magerr_g']
    zr_hjd1,zr_mag1,zr_err1 = tb2['hjd_r'],tb2['mag_r'],tb2['magerr_r']
    zi_hjd1,zi_mag1,zi_err1 = tb3['hjd_i'],tb3['mag_i'],tb3['magerr_i']

    print(f'generating ZTF lcs....\n- remove outliers with sigma > {sigma}')
    zp = 25.
    if len(zg_hjd1)>0:
        jds = Time(zg_hjd1, format='jd', scale='tdb')
        mag = zg_mag1
        magerr = zg_err1
        flux = 10**((mag-zp)/(-2.5))
        flux_err = flux*np.log(10)/(2.5)*magerr # 线性近似
        lczg = lk.LightCurve(time = jds, flux = flux, flux_err = flux_err).remove_outliers(sigma=sigma)
        #lczg.plot()
    else:
        lczg = lk.LightCurve(time = [], flux = [], flux_err = [])
        
    if len(zr_hjd1)>0:    
        jds = Time(zr_hjd1, format='jd', scale='tdb')
        mag = zr_mag1
        magerr = zr_err1
        flux = 10**((mag-zp)/(-2.5))
        flux_err = flux*np.log(10)/(2.5)*magerr # 线性近似
        lczr = lk.LightCurve(time = jds, flux = flux, flux_err = flux_err).remove_outliers(sigma=sigma) 
        #lczr.plot()
    else:
        lczr = lk.LightCurve(time = [], flux = [], flux_err = [])
    if len(zi_hjd1)>0:    
        jds = Time(zi_hjd1, format='jd', scale='tdb')
        mag = zi_mag1
        magerr = zi_err1
        flux = 10**((mag-zp)/(-2.5))
        flux_err = flux*np.log(10)/(2.5)*magerr # 线性近似
        lczi = lk.LightCurve(time = jds, flux = flux, flux_err = flux_err).remove_outliers(sigma=sigma) 
        #lczi.plot()
    else:
        lczi = lk.LightCurve(time = [], flux = [], flux_err = [])
        
    return (lczg, lczr, lczi)

def gen_ztf_lc_mag(dataztf):
    # 输入读取的ztf数据
    # 返回lightkurve光变曲线(lczg, lczr, lczi)
    tb1,tb2,tb3 = dataztf
    zg_hjd1,zg_mag1,zg_err1 = tb1['hjd_g'],tb1['mag_g'],tb1['magerr_g']
    zr_hjd1,zr_mag1,zr_err1 = tb2['hjd_r'],tb2['mag_r'],tb2['magerr_r']
    zi_hjd1,zi_mag1,zi_err1 = tb3['hjd_i'],tb3['mag_i'],tb3['magerr_i']
    
    if len(zg_hjd1)>0:
        jds = Time(zg_hjd1, format='jd', scale='tdb')
        mag = zg_mag1
        magerr = zg_err1
        lczg = TimeSeries(time=jds, data={"mag_zg":mag, "mag_err":magerr})
    else:
        lczg = []
        
    if len(zr_hjd1)>0:    
        jds = Time(zr_hjd1, format='jd', scale='tdb')
        mag = zr_mag1
        magerr = zr_err1
        lczr = TimeSeries(time=jds, data={"mag_zr":mag, "mag_err":magerr})
    else:
        lczr = []
    if len(zi_hjd1)>0:    
        jds = Time(zi_hjd1, format='jd', scale='tdb')
        mag = zi_mag1
        magerr = zi_err1
        lczi = TimeSeries(time=jds, data={"mag_zi":mag, "mag_err":magerr})
    else:
        lczi = []
        
    return (lczg, lczr, lczi)

def gen_asas_lc(dataasas, sigma=150):
    # 输入读取的asassn数据
    # 返回lightkurve光变曲线和asassn header字典(lcV, lcg, header)
    dataV, datag, header = dataasas

    print(f'generating ASASSN lcs....')

    zp = 25.
    if len(dataV)>0:
        jds = Time(np.array(dataV['jd']), format='jd', scale='tdb')
        flux = np.array(dataV['flux'])
        flux_err = np.array(dataV['flux_err'])
        lcV = lk.LightCurve(time = jds, flux = flux, flux_err = flux_err).remove_outliers(sigma=sigma)
        ax = lcV.plot()
        ttl = f'asas_id:{header["asas_sn_id"]}  sdss_id:{header["sdss_id"]}  Gmag:{round(float(header["gaia_mag"]),2)}'+\
        f'    V-band n: {len(dataV)}'
        ax.set_title(ttl)
    else:
        lcV = lk.LightCurve(time = [], flux = [], flux_err = [])
        
    if len(datag)>0:
        jds = Time(np.array(datag['jd']), format='jd', scale='tdb')
        flux = np.array(datag['flux'])
        flux_err = np.array(datag['flux_err'])
        lcg = lk.LightCurve(time = jds, flux = flux, flux_err = flux_err).remove_outliers(sigma=sigma)
        ax = lcg.plot()
        ttl = f'asas_id:{header["asas_sn_id"]}  sdss_id:{header["sdss_id"]}  Gmag:{round(float(header["gaia_mag"]),2)}'+\
        f'    g-band n: {len(datag)}'
        ax.set_title(ttl)
    else:
        lcg = lk.LightCurve(time = [], flux = [], flux_err = [])
        
    return (lcV, lcg, header)

In [10]:
def cal_LombScargle_fromlc(lc, window_length=9991,break_tolerance=5, 
                           tmin=80*u.min, tmax=1*u.day, sample_peak=50, probabilities=[0.01,0.001,0.0005,0.0001],cut_lo=0.01,cut_hi=0.02,
                           ifshow=False, save_path='./lsp.png'):
    # 输入要进行周期图分析的光变曲线lc，进行flatten操作的窗口宽度，
    # 分析的最小和最大周期tmin和tmax，采样点数sample_peak，FAL概率列表(默认选择第二个绘图)probabilities，
    # 周期图图片保存路径save_path
    # 返回(period, best_frequency, frequency, power, probabilities, Alarm_level, lc_flatten, LSperiodogram)，
    # 最佳周期，最佳频率，周期图频率，周期图功率，FAL概率列表，对应FAL水平列表，分析的光变曲线，从光变曲线生成的LombScargle对象
    
    # 如果最小流量值小于0，则将流量向上平移
    if min(lc['flux'])<0:
        lc['flux']=lc['flux']-min(lc['flux'])*1.1
        print('lc min flux <0 has been corrected.')
        
    ax = lc.plot()
    ax.set_title(f'input light curve for Lomb-Scargle Periodogram analysis  [data points {len(lc)}]')
    fig = ax.get_figure()
    fig.savefig(save_path.split('.png')[-2]+'_lc.png', bbox_inches = 'tight', dpi=300)
    plt.close(fig)
    
    print('flatten lc with window_length =',window_length,'  break_tolerance = ',break_tolerance)
    lc_flatten, lc_trend = lc.flatten(window_length=window_length,return_trend=True,break_tolerance=break_tolerance)
    # 保存展平后的光变曲线图片
    ax = lc_flatten.plot()
    ax.set_title(f'flattened light curve for Lomb-Scargle Periodogram analysis  [data points {len(lc_flatten)}]')
    fig = ax.get_figure()
    fig.savefig(save_path.split('.png')[-2]+'_lcflatten.png', bbox_inches = 'tight', dpi=300)
    plt.close(fig)
    # 保存去除的趋势光变曲线图片
    ax = lc_trend.plot()
    ax.set_title(f'the removed trend of the input light curve with window length = {window_length}  '+\
                 f'break: {break_tolerance}  [data points {len(lc_trend)}]',fontsize=9)
    fig = ax.get_figure()
    fig.savefig(save_path.split('.png')[-2]+'_lctrend.png', bbox_inches = 'tight', dpi=300)
    plt.close(fig)
    
    print('>> Calculating Lomb-Scargle Periodogram....')
    LSperiodogram = LombScargle.from_timeseries(lc_flatten, 'flux')
    # 计算周期图
    min_period = tmin
    max_period = tmax
    print('LC data point number:', len(lc_flatten.time))
    print('Flattened lc with window length:',window_length)
    print("min f", np.round(1/max_period.to(u.day),6),   "\tmax f", np.round(1/min_period.to(u.day),6))
    print("max p", np.round(max_period.to(u.hr),3), "\t\tmin p", np.round(min_period.to(u.hr),3))
    print('samples per peak: %d'%sample_peak)
    
    frequency, power = LSperiodogram.autopower(minimum_frequency=1/max_period, maximum_frequency=1/min_period, 
                                               samples_per_peak=sample_peak)
    Alarm_level = LSperiodogram.false_alarm_level(probabilities)
    alarm_select_index = 1 # 设置绘图的FAL, 默认绘制第二个
    
    # 去掉两边
    print(f'* Ignoring {cut_lo*100:.1f} % / {cut_hi*100:.1f} % of each side of LSP.')
    cut = int(len(frequency)*cut_hi)
    cut1 = int(len(frequency)*cut_lo)
    frequency1, power1 = frequency[cut1:len(frequency)-cut], power[cut1:len(frequency)-cut]
    best_frequency = frequency1[np.argmax(power1)]
    period=1/best_frequency
    max_power = max(power1)

    print('f points:',len(frequency))
    print('f step:', round(frequency[3].value-frequency[2].value,7))
    pstep = 1/frequency[9]-1/frequency[10]
    print(f'p step: {pstep.value:.7f} d    {pstep.value*24:.6f} hr    {pstep.value*24*60:.6f} min    {pstep.value*24*3600:.6f} sec', ) 

    f = plt.figure(figsize=(7.2,4.8), dpi=300)
    plt.plot(frequency, power, linewidth=.6)   
    plt.xlim(1/max_period.to(u.day).value, 1/min_period.to(u.day).value)
    plt.plot(best_frequency, max_power, color='red', marker='*',markersize=14, linewidth=0.7)
    plt.xlabel(r'Frequency / $\rm d^{-1}$',size=14)
    plt.ylabel('Amplitude',size=14)
    plt.xticks(size=12)
    plt.yticks(size=12)
    plt.title(f'P_range: {tmin.to(u.day).value:.5f} d - {tmax.to(u.day).value:.5f} d  '+\
              f'FAL_prob={probabilities[alarm_select_index]}  max_power/FAL={max_power.value/Alarm_level[alarm_select_index]:.2f}', size=10)
    plt.text(1/max_period.to(u.day).value + 0.64 * (1/min_period.to(u.day).value-1/max_period.to(u.day).value),
             min(power) + 0.96 * (max(power)-min(power)),
             r'$P_{\rm max}$ = '+str(round(period.to(u.hr).value,6))+' h',size=12)
    plt.axhline(y=Alarm_level[alarm_select_index], ls=":", c="gray", linewidth=0.7, alpha=0.9)
    plt.axvline(x=frequency[cut1].value, ls="dashed", c="grey", linewidth=1, alpha=0.4)
    plt.axvline(x=frequency[len(frequency)-cut].value, ls="dashed", c="grey", linewidth=1, alpha=0.4)
    plt.text(1/max_period.to(u.day).value + 0.78 * (1/min_period.to(u.day).value-1/max_period.to(u.day).value),
            Alarm_level[alarm_select_index] + 0.007 * max(power),
             'FAL: '+str(probabilities[alarm_select_index]), fontsize=9, alpha=0.88, color='gray')
    
    print('Plot FAL - Probability', probabilities[alarm_select_index], '/ Level', round(Alarm_level[alarm_select_index],6))
    print('>>> Best Frequency: ',np.round(best_frequency,7))
    print('>>> Best Period: ', np.round(period,9), ' ', np.round(period.to(u.hr),7), ' ', np.round(period.to(u.min),7))
    print(f'>>> Max Power: {max_power:.5f}   Power/FAL = {max_power/Alarm_level[alarm_select_index]:.3f}')
    
    tbl = Table(names=['best_f(day-1)','best_p(day)','best_p(hr)','max_power',\
                       'FAL_prob1','FAL1','Power_FAL1_Ratio','FAL_prob2','FAL2','Power_FAL2_Ratio','FAL_prob3','FAL3','Power_FAL3_Ratio',\
                       'max_p(day)','min_p(day)','min_f(day-1)','max_f(day-1)','samples_per_peak','p_step(day)','freq_length'],\
                dtype=['f8','f8','f8','f8','f8','f8','f8','f8','f8','f8','f8','f8','f8', 'f8','f8','f8','f8','i4','f8','i4'])
    tbl.add_row([best_frequency,period,period.to(u.hr),max_power,\
                 Alarm_level[1],probabilities[1],max_power/Alarm_level[1],\
                 Alarm_level[2],probabilities[2],max_power/Alarm_level[2],
                 Alarm_level[3],probabilities[3],max_power/Alarm_level[3],\
                max_period.to(u.day),min_period.to(u.day),1/max_period.to(u.day),1/min_period.to(u.day),sample_peak,pstep,len(frequency)])
    tbl.write(save_path.replace('.png','.csv'),format='ascii.csv',overwrite=True)
    print('LSP results have been saved to', save_path.replace('.png','.csv'))
    
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches = 'tight')
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all')
    
    return (period, best_frequency, frequency1, power1, probabilities, Alarm_level, lc_flatten, lc_trend, LSperiodogram)

def plot_LSmodel(LSP_data, select_period=None, xlim=None, sigma_std=5, save_path='./LSmodel.png', time_step=None, ifshow=False):
    # 输入cal_LombScargle_fromlc()返回的数据，人工指定的周期(None则默认LSP中的最佳周期)，显示时间范围xlim(None为默认显示全部)，
    # 显示的流量范围为几倍标准差（默认为5）, model图片保存路径save_path，
    # 人工指定时间间隔(**注意！指定时间间隔后不能继续用预白化程序**)
    # 函数生成LS模型并绘图
    # 返回建模的光变曲线，LS模型，模型所用的周期和频率(lc_flatten, LSmodel, period, best_frequency)
    period, best_frequency, frequency, power, probabilities, Alarm_level, lc_flatten, lc_trend, LSperiodogram = LSP_data
    
    if select_period != None:
        period = select_period
        best_frequency = 1/select_period

    print('>>> Plot LSP model:')
    maxtime = np.nanmax(lc_flatten.time.jd)
    mintime = np.nanmin(lc_flatten.time.jd)
    print('model time JD:', mintime, '-',maxtime)
    
    model_time = lc_flatten.time
    LSmodel = LSperiodogram.model(model_time, best_frequency)
    #model_time2 = Time(np.arange(mintime,maxtime,period.value/10), format='jd', scale='tdb')
    #LSmodel2 = LSperiodogram.model(model_time2, best_frequency)
    
    f = plt.figure(figsize=(24,3),dpi=600)
    b, = plt.plot(lc_flatten.time.jd, lc_flatten.flux, linewidth=0, marker='o', ms=0.6, mec=None)
    #plt.plot(model_time2.jd, LSmodel2, linewidth=0.4, marker='o', markersize=0.8,label='LS_model')
    if time_step is None:
        a, = plt.plot(lc_flatten.time.jd, LSmodel, linewidth=0.6, marker='o', ms=0.4, mec=None, alpha=0.97)
        print('LSmodel plotted with original times.')
    else:
        print('* plotting continued times')
        model_time1 = np.arange(mintime, maxtime, time_step)
        modelTime = Time(model_time1, format='jd', scale='tdb')
        print('-> continued model time:', mintime, '-',maxtime, f' time step: {time_step:.6f}')
        LSmodel1 = LSperiodogram.model(modelTime, best_frequency)
        a, = plt.plot(model_time1, LSmodel1, linewidth=0.09, marker='', ms=0., mec=None, alpha=0.99)
        print('LSmodel plotted with continued times.')
    if xlim != None:
        try:
            plt.xlim(xlim)
        except:
            print('Error! xlim invalid.')
    plt.xlabel(r'JD',size=12)
    plt.ylabel('Normalized Flux',size=12)
    std = np.nanstd(lc_flatten.flux-1)
    # medianval = np.nanmedian(lc_flatten.flux)
    # upper_range = 5 * std
    # plt.ylim(medianval-upper_range, medianval+upper_range)
    mdamp = 0.5*(np.nanmax(LSmodel)-np.nanmin(LSmodel))
    plt.title(f'Lomb-Scargle model    period = {period.to(u.day):.7f} d = {period.to(u.hr).value:.6f} h    amplitude = {mdamp:.4f}'+\
              f'    lc_std = {std:.4f}',
             fontsize=9)
    plt.legend([b,a],['lc_flatten','LS_model'], fontsize=7, loc='upper right')
    # print('lc std:', std)
    # print('best period:', 1/best_frequency, ' ', (1/best_frequency).to(u.hr),
    #       '  best frequency:', best_frequency)
    plt.savefig(save_path, dpi=600, bbox_inches = 'tight')
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all')
        
    return (lc_flatten, LSmodel, period, best_frequency)

def find_power_peaks(LSP_data, npeak=5, save_path='./find_peaks.png', plot_log=False, ifshow=False):
    # 输入cal_LombScargle_fromlc()返回的数据，目标要找到至少几个峰npeak，结果图片保存路径save_path，LSP是否以log尺度绘制plot_log
    # 函数搜索高度最高的一系列峰值
    # 返回(LSP_data, 目标要找的的峰值数量, 实际找到的峰值数量, 找到的峰值对应的index, 找到的峰值的高度, 
    #     找到的峰值对应的频率, 找到的峰值对应的周期)
    period, best_frequency, frequency, power, probabilities, Alarm_level, lc_flatten, lc_trend, LSperiodogram = LSP_data
    # print(power)
    print('  > finding peaks in LSP....')
    print('target peak number:', npeak)
    height = 5*np.std(power).value
    print("  > find height step 1:", height)
    peaks, properties = scipy.signal.find_peaks(power, height=height, threshold=None)
    print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
    if len(properties['peak_heights'])<npeak:
        height = 4*np.std(power).value
        print("  > find height step 2:", height)
        peaks, properties = scipy.signal.find_peaks(power, height=height, threshold=None)
        print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
        if len(properties['peak_heights'])<npeak:
            height = 3*np.std(power).value
            print("  > find height step 3:", height)
            peaks, properties = scipy.signal.find_peaks(power, height=height, threshold=None)
            print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
            if len(properties['peak_heights'])<npeak:
                height = 2.5*np.std(power).value
                print("  > find height step 4:", height)
                peaks, properties = scipy.signal.find_peaks(power, height=height, threshold=None)
                print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
                if len(properties['peak_heights'])<npeak:
                    height = 2.0*np.std(power).value
                    print("  > find height step 5:", height)
                    peaks, properties = scipy.signal.find_peaks(power, height=height, threshold=None)
                    print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
                    if len(properties['peak_heights'])<npeak:
                        height = 1.5*np.std(power).value
                        print("  > find height step 6:", height)
                        peaks, properties = scipy.signal.find_peaks(power, height=height, threshold=None)
                        print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
                        if len(properties['peak_heights'])<npeak:
                            height = 1.2*np.std(power).value
                            print("  > find height step 7:", height)
                            peaks, properties = scipy.signal.find_peaks(power, height=height, threshold=None)
                            print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
                            if len(properties['peak_heights'])<npeak:
                                height = 1.0*np.std(power).value
                                print("  > find height step 8:", height)
                                peaks, properties = scipy.signal.find_peaks(power, height=height, threshold=None)
                                print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
                                if len(properties['peak_heights'])<npeak:
                                    height = 0.8*np.std(power).value
                                    print("  > find height step 9:", height)
                                    peaks, properties = scipy.signal.find_peaks(power, height=height, threshold=None)
                                    print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
                                    if len(properties['peak_heights'])<npeak:
                                        height = 0.7*np.std(power).value
                                        print("  > find height step 10:", height)
                                        peaks, properties = scipy.signal.find_peaks(power, height=height, threshold=None)
                                        print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
                                        if len(properties['peak_heights'])<npeak:
                                            height = 0.6*np.std(power).value
                                            print("  > find height step 11:", height)
                                            peaks, properties = scipy.signal.find_peaks(power, height=height, 
                                                                                        threshold=None)
                                            print('peaks:',len(peaks), ' peak index:', peaks, '',properties)
    
    # 按峰高度排序
    height_index = np.argsort(properties['peak_heights'])
    height_index = np.flipud(height_index)
    print()
    print('>>> Final result:\nfind peaks:', len(peaks), ' (target number: %2d)'%npeak,
          '\npeak index:', peaks[height_index], 
          '\npeak heights:', properties['peak_heights'][height_index],
          '\npeak frequencies:', frequency[peaks[height_index]],
          '\npeak periods:', 1/frequency[peaks[height_index]])
    
    f = plt.figure(figsize=(6,4), dpi=200)
    plt.plot(frequency, power, linewidth=.6)   
    plt.plot(frequency[peaks], properties['peak_heights'], color='red', marker='*',markersize=5, linewidth=0)

    for i in range(len(peaks)):
        plt.text(frequency[peaks[i]].value+0.0012,
                 properties['peak_heights'][i] + max(power) * 0.01,
                 str(round(1/frequency[peaks[i]].value,6))+' d',
                 size=4.8)
    plt.xlabel(r'Frequency / $\rm d^{-1}$',size=12)
    plt.ylabel('Amplitude',size=12)
    if plot_log:
        plt.yscale('log')
    alarm_select_index = 1
    plt.axhline(y=Alarm_level[alarm_select_index], ls=":", c="black", linewidth=0.7)
    xtext = max(frequency) - 0.20 * (max(frequency)-min(frequency))
    plt.text(xtext.value, Alarm_level[alarm_select_index] + 0.008 * max(power),
             'FAL: '+str(probabilities[alarm_select_index]), fontsize=6.5)
    print()
    print('Plot FAL - Probability:', probabilities[alarm_select_index], '/ Level:', 
          round(Alarm_level[alarm_select_index],6))
    plt.savefig(save_path, dpi=200, bbox_inches = 'tight')
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all')
    
    return (LSP_data, npeak, len(peaks), peaks[height_index], properties['peak_heights'][height_index], 
            frequency[peaks[height_index]], 1/frequency[peaks[height_index]])

def fold_lc_fromLS(LSP_data, save_path='./folded_lc.png', nbin=10, nstd=5, mode='median', errorbar=False, ifshow=False):
    # 输入cal_LombScargle_fromlc()返回的数据，相位折叠图保存路径save_path，降采样点数nbin，显示流量范围为几倍标准差nstd，
    # 降采样模式('median'或'mean')，是否绘制降采样点的误差棒errorbar
    # 函数以LSP数据中的最佳周期折叠光变曲线(*相位随机，不指定*)
    # 返回(折叠所用的最佳周期, 最佳频率, 折叠所用的光变曲线, 折叠后的光变曲线, 折叠后的光变曲线降采样后的光变曲线(内含误差),
    #      折叠后光变曲线一个周期的相位, 折叠后光变曲线一个周期的流量, 
    #      降采样光变曲线一个周期的相位, 降采样光变曲线一个周期的流量, 降采样光变曲线一个周期的流量误差)
    period, best_frequency, frequency, power, probabilities, Alarm_level, lc_flatten, lc_trend, LSperiodogram = LSP_data
    lc_folded = lc_flatten.fold(period = period) 
    print('>>> fold lc with best frequency (epoch not specified) ....')
    lc_folded.time=lc_folded.time+period/2
    lc_folded_phase=lc_folded.time/period
    print('lc data points:',len(lc_folded['flux']),' bin number:',nbin)
    if mode == 'median':
        func = np.nanmedian
        lc_folded_binned = aggregate_downsample(lc_folded, time_bin_size = period/nbin, aggregate_func=func)
    elif mode == 'mean':
        func = np.nanmean
        lc_folded_binned = aggregate_downsample(lc_folded, time_bin_size = period/nbin, aggregate_func=func)
    else:
        raise InputEror("'mode'can only be 'median' or 'mean'!")
        
    print(f'Folded Period: {period.value:.7f} d  {period.to(u.hr).value:.6f} hr  {period.to(u.min).value:.6f} min')        
    f = plt.figure(figsize=(4.8,3),dpi=200)
    plt.plot(lc_folded_phase,   lc_folded['flux'], 'k,', alpha=0.7, zorder=0)
    plt.plot(lc_folded_phase+1, lc_folded['flux'], 'k,', alpha=0.7, zorder=0)
    plt.scatter((lc_folded_binned.time_bin_start.jd/period.value),   lc_folded_binned['flux'].value, 
                marker='.',c='r',s=9)
    plt.scatter((lc_folded_binned.time_bin_start.jd/period.value)+1, lc_folded_binned['flux'].value, 
                marker='.',c='r',s=9)
    amp = 0.5*(np.nanmax(lc_folded_binned['flux'].value)-np.nanmin(lc_folded_binned['flux'].value))    
    if errorbar:
        try:
            plt.errorbar((lc_folded_binned.time_bin_start.jd/period.value),
                 lc_folded_binned['flux'].value, 
                 yerr=lc_folded_binned['flux_err'].value, 
                 color='r', linewidth=0, elinewidth=.4, capsize=1, capthick=.8)
            plt.errorbar((lc_folded_binned.time_bin_start.jd/period.value)+1,
                 lc_folded_binned['flux'].value, 
                 yerr=lc_folded_binned['flux_err'].value, 
                 color='r', linewidth=0, elinewidth=.4, capsize=1, capthick=.8)
        except:
            print('* yerr has nagative value.')
            plt.errorbar((lc_folded_binned.time_bin_start.jd/period.value),
                 lc_folded_binned['flux'].value, 
                 yerr=np.abs(lc_folded_binned['flux_err'].value), 
                 color='r', linewidth=0, elinewidth=.4, capsize=1, capthick=.8)
            plt.errorbar((lc_folded_binned.time_bin_start.jd/period.value)+1,
                 lc_folded_binned['flux'].value, 
                 yerr=np.abs(lc_folded_binned['flux_err'].value), 
                 color='r', linewidth=0, elinewidth=.4, capsize=1, capthick=.8)

    tbl = Table(names=['folded_period(day)','bin_points','bin_time(hr)','semi-amplitude','aggregate_func'],\
                dtype=['f8','i4','f8','f8','S65'])
    tbl.add_row([period.to(u.day),nbin,period.to(u.hr).value/nbin,amp,str(func)])
    tbl.write(save_path.replace('.png','.csv'),format='ascii.csv',overwrite=True)
    print('>> Folded results have been saved to', save_path.replace('.png','.csv'))
    
    std = np.nanstd(lc_flatten.flux)
    upper_range = nstd * std
    plt.ylim(1-upper_range, 1+upper_range)
    plt.text(1.33, 1 + 0.8*upper_range, '$P$='+str(round(period.to(u.hr).value,6))+' h',size=10.5)
    plt.xlabel('Phase',size=12)
    plt.ylabel('Normalized Flux',size=12)
    plt.xticks(np.arange(0,2.1,0.2),size=11)
    plt.yticks(size=11)
    plt.title(f'Data points: {len(lc_folded)}  Bin points: {nbin}  Bin time: {period.to(u.hr).value/nbin:.3f} hr  Amplitude: {amp:.4f}',\
              fontsize=7)
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches = 'tight')
    
    lc_folded_binned_flux = lc_folded_binned['flux']
    binned_amplitude = (np.nanmax(lc_folded_binned_flux)-np.nanmin(lc_folded_binned_flux))/2.
    print('binned_amplitude:',binned_amplitude)
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all')
    
    return (period, best_frequency, lc_flatten, lc_folded, lc_folded_binned,
            lc_folded_phase, lc_folded['flux'], 
            lc_folded_binned.time_bin_start.jd/period.value, lc_folded_binned['flux'], lc_folded_binned['flux_err'])

def prewhiten(LSmodel_data, save_path=None, ifshow=False):
    # 输入plot_LSmodel()返回的数据
    # 函数通过将原始光变曲线流量除以model光变曲线流量，将model光变曲线的周期剔除，并展示光变曲线的三个部分
    # 返回剔除了model光变曲线周期的光变曲线，以重新进行周期图计算(cal_LombScargle_fromlc)
    
    lc_flatten, LSmodel, period, best_frequency = LSmodel_data
    
    sigma_std = 5
    std = np.std(lc_flatten.flux-1)
    
    print('>>> Calculate residual lightcurve...')
    print('Eliminate period =', period)
    f = plt.figure(figsize=(10,3),dpi=300)
    resi_flux = lc_flatten.flux/LSmodel
    plt.plot(lc_flatten.time.jd, lc_flatten.flux, linewidth=0, marker='o', markersize=1)
    plt.plot(lc_flatten.time.jd, LSmodel, linewidth=0.6, marker='o', markersize=1)
    plt.plot(lc_flatten.time.jd, resi_flux, linewidth=0.5, marker='o', markersize=0, c='limegreen')
    if len(lc_flatten.time.jd)>2000:
        split = len(lc_flatten.time.jd)//10
        plt.xlim(lc_flatten.time.jd[split],lc_flatten.time.jd[split*2])
    elif len(lc_flatten.time.jd)>1500:
        split = len(lc_flatten.time.jd)//9
        plt.xlim(lc_flatten.time.jd[split],lc_flatten.time.jd[split*2])
    elif len(lc_flatten.time.jd)>1000:
        split = len(lc_flatten.time.jd)//5
        plt.xlim(lc_flatten.time.jd[split],lc_flatten.time.jd[split*2])
    elif len(lc_flatten.time.jd)>600:
        split = len(lc_flatten.time.jd)//4
        plt.xlim(lc_flatten.time.jd[0],lc_flatten.time.jd[split])
    plt.ylim(-sigma_std*std+1, sigma_std*std+1)
    plt.xlabel(r'JD',size=12)
    plt.ylabel('Normalized Flux',size=12)
    plt.title('prewhiten check part-1')
    plt.legend(['LC','model','residual'], fontsize=7, loc='upper right')
    if save_path is not None:
        plt.savefig(save_path.split('.png')[-2]+'_p1.png', dpi=300, bbox_inches = 'tight')
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all') 
    
    f = plt.figure(figsize=(10,3),dpi=300)
    resi_flux = lc_flatten.flux/LSmodel
    plt.plot(lc_flatten.time.jd, lc_flatten.flux, linewidth=0, marker='o', markersize=1)
    plt.plot(lc_flatten.time.jd, LSmodel, linewidth=0.6, marker='o', markersize=1)
    plt.plot(lc_flatten.time.jd, resi_flux, linewidth=0.5, marker='o', markersize=0, c='limegreen')
    if len(lc_flatten.time.jd)>2000:
        split = len(lc_flatten.time.jd)//10
        plt.xlim(lc_flatten.time.jd[split*2],lc_flatten.time.jd[split*3])
    elif len(lc_flatten.time.jd)>1500:
        split = len(lc_flatten.time.jd)//9
        plt.xlim(lc_flatten.time.jd[split*2],lc_flatten.time.jd[split*3])
    elif len(lc_flatten.time.jd)>1000:
        split = len(lc_flatten.time.jd)//5
        plt.xlim(lc_flatten.time.jd[split*2],lc_flatten.time.jd[split*3])
    elif len(lc_flatten.time.jd)>600:
        split = len(lc_flatten.time.jd)//4
        plt.xlim(lc_flatten.time.jd[split],lc_flatten.time.jd[split*2])
    plt.ylim(-sigma_std*std+1, sigma_std*std+1)
    plt.xlabel(r'JD',size=12)
    plt.ylabel('Normalized Flux',size=12) 
    plt.title('prewhiten check part-2')
    plt.legend(['LC','model','residual'], fontsize=7, loc='upper right')
    if save_path is not None:
        plt.savefig(save_path.split('.png')[-2]+'_p2.png', dpi=300, bbox_inches = 'tight')
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all') 
    
    f = plt.figure(figsize=(10,3),dpi=300)
    resi_flux = lc_flatten.flux/LSmodel
    plt.plot(lc_flatten.time.jd, lc_flatten.flux, linewidth=0, marker='o', markersize=1)
    plt.plot(lc_flatten.time.jd, LSmodel, linewidth=0.6, marker='o', markersize=1)
    plt.plot(lc_flatten.time.jd, resi_flux, linewidth=0.5, marker='o', markersize=0, c='limegreen')
    if len(lc_flatten.time.jd)>2000:
        split = len(lc_flatten.time.jd)//10
        plt.xlim(lc_flatten.time.jd[split*8],lc_flatten.time.jd[split*9])
    elif len(lc_flatten.time.jd)>1500:
        split = len(lc_flatten.time.jd)//9
        plt.xlim(lc_flatten.time.jd[split*7],lc_flatten.time.jd[split*8])
    elif len(lc_flatten.time.jd)>1000:
        split = len(lc_flatten.time.jd)//5
        plt.xlim(lc_flatten.time.jd[split*3],lc_flatten.time.jd[split*4])
    elif len(lc_flatten.time.jd)>600:
        split = len(lc_flatten.time.jd)//4
        plt.xlim(lc_flatten.time.jd[split*2],lc_flatten.time.jd[split*3])
    plt.ylim(-sigma_std*std+1, sigma_std*std+1)
    plt.xlabel(r'JD',size=12)
    plt.ylabel('Normalized Flux',size=12) 
    plt.title('prewhiten check part-3')
    plt.legend(['LC','model','residual'], fontsize=7, loc='upper right')
    if save_path!=None:
        plt.savefig(save_path.split('.png')[-2]+'_p3.png', dpi=300, bbox_inches = 'tight')
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all')    
    
    resi_lc = lk.LightCurve(time = lc_flatten.time, flux = resi_flux, flux_err = lc_flatten['flux_err']/LSmodel)
    
    return resi_lc

def fold_lc_fromLS_epoch(LSP_data, epoch_time, save_path=None, nbin=9, nstd=5, ifshow=False, mode='median', errorbar=False):
    # 输入cal_LombScargle_fromlc()返回的数据，相位折叠图保存路径save_path，降采样点数nbin，显示流量范围为几倍标准差nstd，
    # 降采样模式('median'或'mean')，是否绘制降采样点的误差棒errorbar
    # 函数以LSP数据中的最佳周期折叠光变曲线
    # 返回(折叠所用的最佳周期, 最佳频率, 折叠所用的光变曲线, 折叠后的光变曲线, 折叠后的光变曲线降采样后的光变曲线(内含误差),
    #      折叠后光变曲线一个周期的相位, 折叠后光变曲线一个周期的流量, 
    #      降采样光变曲线一个周期的相位, 降采样光变曲线一个周期的流量, 降采样光变曲线一个周期的流量误差)
    period, best_frequency, frequency, power, probabilities, Alarm_level, lc_flatten, lc_trend, LSperiodogram = LSP_data
    lc_folded = lc_flatten.fold(period = period, epoch_time=epoch_time) 
    print(f'>>> fold lc with best frequency (epoch is specified to {epoch_time})....')
    lc_folded.time=lc_folded.time+period/2
    lc_folded_phase=lc_folded.time/period
    print('lc data points:',len(lc_folded['flux']),' bin number:',nbin)
    if mode == 'median':
        func = np.nanmedian
        lc_folded_binned = aggregate_downsample(lc_folded, time_bin_size = period/nbin, aggregate_func=func)
    elif mode == 'mean':
        func = np.nanmean
        lc_folded_binned = aggregate_downsample(lc_folded, time_bin_size = period/nbin, aggregate_func=func)
    else:
        raise InputEror("'mode'can only be 'median' or 'mean'!")
    amp = 0.5*(np.nanmax(lc_folded_binned['flux'].value)-np.nanmin(lc_folded_binned['flux'].value))            
    print(f'Folded Period: {period.value:.7f} d  {period.to(u.hr).value:.6f} hr  {period.to(u.min).value:.6f} min')        
    f = plt.figure(figsize=(4.8,3),dpi=200)
    plt.plot(lc_folded_phase,   lc_folded['flux'], 'k,', alpha=0.7, zorder=0)
    plt.plot(lc_folded_phase+1, lc_folded['flux'], 'k,', alpha=0.7, zorder=0)
    plt.scatter((lc_folded_binned.time_bin_start.jd/period.value),   lc_folded_binned['flux'].value, 
                marker='.',c='r',s=9)
    plt.scatter((lc_folded_binned.time_bin_start.jd/period.value)+1, lc_folded_binned['flux'].value, 
                marker='.',c='r',s=9)
    
    if errorbar:
        plt.errorbar((lc_folded_binned.time_bin_start.jd/period.value),
             lc_folded_binned['flux'].value, 
             yerr=lc_folded_binned['flux_err'].value, 
             color='r', linewidth=0, elinewidth=.4, capsize=1, capthick=.8)
        plt.errorbar((lc_folded_binned.time_bin_start.jd/period.value)+1,
             lc_folded_binned['flux'].value, 
             yerr=lc_folded_binned['flux_err'].value, 
             color='r', linewidth=0, elinewidth=.4, capsize=1, capthick=.8)
        
    std = np.nanstd(lc_flatten.flux)
    upper_range = nstd * std
    plt.ylim(1-upper_range, 1+upper_range)
    plt.text(1.33, 1 + 0.8*upper_range, '$P$='+str(round(period.to(u.hr).value,6))+' h',size=10.5)
    plt.text(1.25, 1 - 0.90*upper_range, 'Epoch: %.8f'%(epoch_time),size=6.5)
    plt.xlabel('Phase',size=12)
    plt.ylabel('Normalized Flux',size=12)
    plt.xticks(np.arange(0,2.1,0.2),size=11)
    plt.yticks(size=11)
    
    lc_folded_binned_flux = lc_folded_binned['flux']
    print('binned_amplitude:',amp)
    print(f'epoch_time: {epoch_time}')
    plt.title(f'Data points: {len(lc_folded)}  Bin points: {nbin}  Bin time: {period.to(u.hr).value/nbin:.3f} hr  Amplitude: {amp:.4f}',\
              fontsize=7)

    if save_path is not None:
        tbl = Table(names=['folded_period(day)','bin_points','bin_time(hr)','semi-amplitude','aggregate_func'],\
                    dtype=['f8','i4','f8','f8','S65'])
        tbl.add_row([period.to(u.day),nbin,period.to(u.hr).value/nbin,amp,str(func)])
        tbl.write(save_path.replace('.png','.csv'),format='ascii.csv',overwrite=True)
        print('>> Folded results have been saved to', save_path.replace('.png','.csv'))
        plt.savefig(save_path, dpi=200, bbox_inches = 'tight')
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all')
    
    return (period, best_frequency, lc_flatten, lc_folded, lc_folded_binned,
            lc_folded_phase, lc_folded['flux'], 
            lc_folded_binned.time_bin_start.jd/period.value, lc_folded_binned['flux'], lc_folded_binned['flux_err'])

In [11]:
import scipy.interpolate as spint

# 确定最小值星历
def determine_epoch(LSP_data,nbin=15,nbin2=11,nstd=5,smooth_factor=0.0002,save_path=None,ifshow=False):
    # 输入cal_LombScargle_fromlc()返回的数据
    # 函数先按照第一个数据点的时刻为参考折叠按照最佳周期折叠光变曲线，然后通过样条插值和平滑得到相位周期轮廓，
    # 然后确定相位周期轮廓的极小值对应的相位，最后计算修正的参考时间
    # 返回LSP数据中的周期(折叠所用周期)，第一个数据点时刻为参考对应的极小相位，
    # 计算出相对于第一个数据点的时刻的时间改正量，改正后的参考时间new_epoch，
    # 以及根据new_epoch进行折叠以后的相位折叠光变曲线数据（参考fold_lc_fromLS_epoch()函数）:
    # (period, min_phase, epoch_time0, time_shift, new_epoch, newfolddata, min_times)
    
    # 目的是让光变曲线极小值在相位0时刻
    # 指定一个epoch，生成相位折叠轮廓
    # 利用相位折叠轮廓生成插值轮廓
    # 计算插值轮廓最小值的相位
    # 计算插值轮廓最小值对应的时间偏移
    # 修正时间偏移，得到最小值的预测
    # 将最小值的预测绘制到原始光变曲线上
    # 返回周期、选定的极小值时刻、预测的极小值位置
    period, best_frequency, frequency, power, probabilities, Alarm_level, lc_flatten, lc_trend, LSperiodogram = LSP_data
    
    epoch_time0 = lc_flatten.time.jd[0]
    print('lc start time:',epoch_time0)
    if save_path != None:
        save_path2 = save_path.split('.png')[-2]+'_PhaseCorrected.png'
        save_path1 = None
    else:
        save_path2, save_path1 = None, None
    mode='median'
    errorbar=False
    
    (period, best_frequency, lc_flatten, lc_folded, lc_folded_binned,
    lc_folded_phase, lc_folded_flux, 
    lc_folded_binned_phase, lc_folded_binned_flux, lc_folded_binned_fluxerr) = fold_lc_fromLS_epoch(LSP_data,
                                                                                                epoch_time=epoch_time0,
                                                                                                save_path=save_path1,
                                                                                                nbin=nbin, nstd=nstd,
                                                                                                mode=mode,errorbar=False)
    intep_phase = np.concatenate((lc_folded_binned_phase[-2:-1]-1,lc_folded_binned_phase[-1:]-1,
                                  lc_folded_binned_phase,
                                  lc_folded_binned_phase[:1]+1,lc_folded_binned_phase[1:2]+1))
    intep_flux = np.concatenate((lc_folded_binned_flux[-2:-1], lc_folded_binned_flux[-1:],
                                 lc_folded_binned_flux,
                                 lc_folded_binned_flux[:1],lc_folded_binned_flux[1:2]))
    splt = spint.splrep(intep_phase, intep_flux, s=smooth_factor) # s是平滑因子
    phases = np.linspace(0,1,1000001)  # 精度1e-6
    profile = spint.splev(phases,splt,der=0)
    minimum_index = np.argmin(profile)
    min_phase = phases[minimum_index]
    print('original minimum phase:',np.round(min_phase,6))
    time_shift = period*min_phase+1/2*period
    new_epoch = epoch_time0 + time_shift.value
    print('>> corrected Epoch:', new_epoch)
    
    f = plt.figure(figsize=(4.5,3),dpi=200)
    std = np.nanstd(lc_flatten.flux)
    
    plt.plot(phases,  profile,color='tab:blue',alpha=0.9)
    plt.plot(phases+1,profile,color='tab:blue',alpha=0.9)
    plt.plot(np.linspace(0,2,100),np.ones(100)*1,linestyle='dotted',linewidth=0.6,color='#020202')
    plt.plot(np.ones(10)*min_phase,    np.linspace(1-std,1+std,10),
             linestyle='dotted',linewidth=0.9,color='#020202')
    plt.plot(np.ones(10)*(min_phase+1),np.linspace(1-std,1+std,10),
             linestyle='dotted',linewidth=0.9,color='#020202')
    plt.plot(lc_folded_phase,   lc_folded['flux'], 'k,', alpha=0.7, zorder=0)
    plt.plot(lc_folded_phase+1, lc_folded['flux'], 'k,', alpha=0.7, zorder=0)
    plt.scatter(lc_folded_binned_phase,   lc_folded_binned_flux.value, 
                marker='.',c='r',s=9)
    plt.scatter(lc_folded_binned_phase+1, lc_folded_binned_flux.value, 
                marker='.',c='r',s=9)
    amp = 0.5*(np.nanmax(lc_folded_binned_flux.value)-np.nanmin(lc_folded_binned_flux.value))
    upper_range = nstd * std
    plt.ylim(1-upper_range, 1+upper_range)
    plt.text(1.4, 1 + 0.8*upper_range, '$P$='+str(round(period.to(u.hr).value,6))+' h',size=9.0)
    plt.text(1.35, 1 + 0.68*upper_range, '='+str(round(period.to(u.day).value,8))+' d',size=9.0)
    plt.text(1.25, 1 - 0.80*upper_range, 'minimum phase: %.6f'%(min_phase),size=6.5)
    plt.text(1.25, 1 - 0.90*upper_range, 'Epoch: %.8f'%(new_epoch),size=6.5)
    plt.xticks(np.arange(0,2.1,0.2),size=9.6)
    plt.yticks(size=9.6)
    plt.xlabel('Phase',size=11)
    plt.ylabel('Normalized Flux',size=11)
    plt.title(f'Correct Epoch - Bin points: {nbin2}  Bin time: {period.to(u.hr).value/nbin2:.3f} hr  Amplitude: {amp:.4f}',fontsize=7)
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches = 'tight')
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all')
    
    newfolddata = fold_lc_fromLS_epoch(LSP_data,epoch_time=new_epoch,
                                       save_path=save_path2,nbin=nbin2, nstd=nstd,mode=mode,errorbar=False)
    print(f'>> Ephemeries: JD(minimum) = {new_epoch} + {period.value} * E')
    min_times = []
    mintime = new_epoch
    for i in range(1001):
        min_times.append(mintime)
        mintime += period.value
    
    return (period, min_phase, epoch_time0, time_shift, new_epoch, newfolddata, min_times)

def generate_new_lc_Monte_Carlo(original_lc):
    # 输入原始光变曲线，包含时间、流量、流量误差
    # 函数根据流量和流量误差按照高斯分布生成新的流量
    # 返回新生成的光变曲线，重复调用该函数就可进行蒙特卡洛误差分析
    time = original_lc.time
    data_length = len(time)
    randomdata = np.random.normal(loc=original_lc['flux'].value, scale=original_lc['flux_err'].value)
    gen_lc = lk.LightCurve(time = time, flux = randomdata)
    return gen_lc

# tesslc ='./RUN/20240626_test1/000007_J065237.19+243622.2_LAMOST J065237.19+243622.1/data/lc/_J065237.19+243622.2_06_TESS Sector 71_SPOC_120.csv'
# datatess = read_tess(tesslc)
# lc_btjd, lc_jd = gen_tess_lc(datatess, sigma=9)
# lspdata = cal_LombScargle_fromlc(lc_jd, window_length=9999, 
#                                 tmin=15*u.min, tmax=1*u.day, sample_peak=45, probabilities=[0.01,0.001,0.0001,0.00001], # 默认绘制第二个概率
#                                 save_path='./Temp/test1_S01_LSP.png')
# epochfolddata = determine_epoch(lspdata,nbin=15,nbin2=21,smooth_factor=0.00002,
#                                            save_path='./Temp/test1_S04_PhaseProfile.png')
# period_e, min_phasee, epoch_time0, time_shift, new_epoch, newfolddata, min_times = epochfolddata

In [12]:
def cal_LSP_from_ts(ts, col_name, tmin, tmax, samples_factor=5, cut_lo=0.0185, cut_hi=0.02,
                    probabilities=[0.05, 0.01, 0.005, 0.001], level_select=1, 
                    save_name='./LSP_from_ts.png', ifshow=False):
    print('*** Calculating Lomb-Scargle Periodogram.... ***')
    LSperiodogram = LombScargle.from_timeseries(ts, col_name)
    min_period = tmin
    max_period = tmax
    spp = int((max_period.to(u.day).value/10*samples_factor)//2*2 + 1)

    print("min f",np.round(1/max_period.to(u.day),6),"   max f",np.round(1/min_period.to(u.day),6))
    print("max p",max_period.to(u.day),"       min p",min_period.to(u.min))
    print('samples_per_peak: %d'%spp)
    
    frequency, power = LSperiodogram.autopower(minimum_frequency=1/max_period,maximum_frequency=1/min_period,
                                               samples_per_peak=spp)
    # 去掉两边
    print(f'* Ignoring {cut_lo*100:.1f} / {cut_hi*100:.1f}% of each side of LSP.')
    cut = int(len(frequency)*cut_hi)
    cut1 = int(len(frequency)*cut_lo)
    frequency1, power1 = frequency[cut1:len(frequency)-cut], power[cut1:len(frequency)-cut]
    best_frequency = frequency1[np.argmax(power1)]
    period=1/best_frequency
    Alarm_level = LSperiodogram.false_alarm_level(probabilities)
    max_power = max(power1)
    
    print('f_length:', len(frequency))
    print('f_step:', round(frequency[3].value-frequency[2].value,7))
    pstep = 1/frequency[9]-1/frequency[10]
    print(f'p step: {pstep.value:.7f} d    {pstep.value*24:.6f} hr    {pstep.value*24*60:.5f} min', )    
    
    f=plt.figure(figsize=(5,3),dpi=240)
    plt.plot(frequency, power, linewidth=0.4)   
    plt.xlim(1/max_period.to(u.day).value,1/min_period.to(u.day).value)
    plt.plot(best_frequency,max_power,color='red',marker='*',markersize=10)
    plt.xlabel(r'Frequency / $\rm d^{-1}$',size=9)
    plt.ylabel('Amplitude',size=9)
    plt.xticks(size=9)
    plt.yticks(size=9)
    plt.title(f'FAL prob={probabilities[level_select]}  max_power/FAL={np.round(max_power.value/Alarm_level[level_select],2)}', size=10)
    plt.text(1/max_period.to(u.day).value + 0.47 * (1/min_period.to(u.day).value-1/max_period.to(u.day).value),
             min(power) + 0.97 * (max(power)-min(power)),
             r'$P_{\rm max}$='+str(round(period.to(u.hr).value,7))+' h='+\
             str(round(period.to(u.day).value,5))+' d',size=8)

    plt.axhline(y=Alarm_level[level_select], ls=":", c="black", linewidth=0.7)
    plt.axvline(x=frequency[cut1].value, ls="dashed", c="grey", linewidth=1, alpha=0.4)
    plt.axvline(x=frequency[len(frequency)-cut].value, ls="dashed", c="grey", linewidth=1, alpha=0.4)
    plt.text(1/max_period.to(u.day).value + 0.78 * (1/min_period.to(u.day).value-1/max_period.to(u.day).value),
            Alarm_level[level_select] + 0.008 * max(power),
             'FAL: '+str(probabilities[level_select]), fontsize=8)
    
    print('Probability',probabilities[level_select],'/ Level',round(Alarm_level[level_select],3))
    print('>> Best Frequency: ',np.round(best_frequency,7))
    print('>> Best Period: ', np.round(period,9), ' ', np.round(period.to(u.hr),7), ' ', np.round(period.to(u.min),7))
    print(f'>>> Max Power: {max_power:.5f}   Power/FAL = {max_power/Alarm_level[level_select]:.3f}')

    if save_name is not None:
        tbl = Table(names=['best_f(day-1)','best_p(day)','best_p(hr)','max_power',\
                           'FAL_prob1','FAL1','Power_FAL1_Ratio','FAL_prob2','FAL2','Power_FAL2_Ratio','FAL_prob3','FAL3','Power_FAL3_Ratio',\
                           'max_p(day)','min_p(day)','min_f(day-1)','max_f(day-1)','samples_per_peak','p_step(day)','freq_length'],\
                    dtype=['f8','f8','f8','f8','f8','f8','f8','f8','f8','f8','f8','f8','f8', 'f8','f8','f8','f8','i4','f8','i4'])
        tbl.add_row([best_frequency,period,period.to(u.hr),max_power,\
                     Alarm_level[1],probabilities[1],max_power/Alarm_level[1],\
                     Alarm_level[2],probabilities[2],max_power/Alarm_level[1],
                     Alarm_level[3],probabilities[3],max_power/Alarm_level[1],\
                    max_period.to(u.day),min_period.to(u.day),1/max_period.to(u.day),1/min_period.to(u.day),spp,pstep,len(frequency)])
        tbl.write(save_name.replace('.png','.csv'),format='ascii.csv',overwrite=True)
        print('LSP results have been saved to', save_name.replace('.png','.csv'))
        plt.savefig(save_name, dpi=240, bbox_inches = 'tight')
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all')
    
    return (period, best_frequency, frequency, power, probabilities, Alarm_level, ts, LSperiodogram)
    
def f_sin0(x, amplitude, constant, phase):
    return amplitude*np.sin(2*np.pi*(x+phase))+constant
    
def fold_ts(ts, col_name, period, epoch_time=None, bin_size=9, func=np.nanmedian, fit=True, dot=False, save_name='./folded_ts.png',ifshow=False,
           invert_y=False):
    period = period.to(u.day)
    if epoch_time is None:
        ts_folded = ts.fold(period = period) 
        epoch_time = ts.time[0]
    else:
        ts_folded = ts.fold(period = period, epoch_time=epoch_time) 

    ts_folded_phase = ts_folded.time/period
    ts_folded_binned = aggregate_downsample(ts_folded, time_bin_size=period/bin_size, aggregate_func=func)
    print(f'Folded Period: {period.value:.7f} d  {period.to(u.hr).value:.6f} hr  {period.to(u.min).value:.6f} min')   
    f=plt.figure(figsize=(5,3),dpi=240)
    if not dot:
        plt.scatter(ts_folded_phase+0.5, ts_folded[col_name], c='k', marker='x', s=9, alpha=0.5, zorder=0)
        plt.scatter(ts_folded_phase+1.5, ts_folded[col_name], c='k', marker='x', s=9, alpha=0.5, zorder=0)
    else:
        plt.plot(ts_folded_phase+0.5, ts_folded[col_name], 'k,', alpha=0.7, zorder=0)
        plt.plot(ts_folded_phase+1.5, ts_folded[col_name], 'k,', alpha=0.7, zorder=0)     
    binned_phase = (ts_folded_binned.time_bin_start.jd/period).value
    binned_data = ts_folded_binned[col_name]
    amp = 0.5*(max(binned_data)-min(binned_data))
    print('Binned size:',bin_size)
    print('Folded amplitude:', str(np.round(amp,5)))
    plt.scatter(binned_phase+0.5, binned_data.value, marker='.',c='r',s=9, zorder=2)
    plt.scatter(binned_phase+1.5, binned_data.value, marker='.',c='r',s=9, zorder=2)
    try:
        binned_data1 = binned_data.value[np.logical_not(np.isnan(binned_data.value.filled(np.nan)))]
        binned_phase1 = binned_phase[np.logical_not(np.isnan(binned_data.value.filled(np.nan)))]
        binned_phase2, binned_data2 = np.concatenate([binned_phase1+0.5,binned_phase1+1.5]), np.concatenate([binned_data1,binned_data1])
    except:
        binned_data1 = binned_data.copy()
        binned_phase1 = binned_phase.copy()
        binned_phase2, binned_data2 = np.concatenate([binned_phase1+0.5,binned_phase1+1.5]), np.concatenate([binned_data1,binned_data1])        
    #plt.plot(binned_phase2, binned_data2, 'r-', drawstyle='steps-pre', linewidth=0.25,zorder=2,alpha=0.7)

    if fit and len(binned_data)>5:
        fit_pass = True
        print('fitting sin curve for binned data...')
        try:           
            phase0 = binned_phase2[np.argmax(binned_data2)]-0.5
            if phase0<0:
                phase0 = 1-phase0
            popt, pcov = optimize.curve_fit(f_sin0, binned_phase2, binned_data2, 
                                            p0=[1.12*amp,0.5*(np.nanmax(binned_data)+np.nanmin(binned_data)),phase0],
                                            bounds=[[0.2*amp,-np.inf,0],[np.inf,np.inf,1]])
        except:
            print('fit failed. retrying...')
            try:
                popt, pcov = optimize.curve_fit(f_sin0, binned_phase2, binned_data2, 
                                            p0=[1.25*amp,0.5*(np.nanmax(binned_data)+np.nanmin(binned_data)),phase0],
                                            bounds=[[0.2*amp,-np.inf,0],[np.inf,np.inf,1]])
            except:
                print('fit failed. retrying...')
                try:
                    popt, pcov = optimize.curve_fit(f_sin0, binned_phase2, binned_data2,
                                                    p0=[1.5*amp,0.5*(np.nanmax(binned_data)+np.nanmin(binned_data)), 0.8],
                                                    bounds=[[0.5*amp,-np.inf,0],[np.inf,np.inf,1]])
                except:
                    print('unable to fit a sin curve.')
                    fit_pass = False
        if fit_pass:
            print('fit success.')
            print('>> sin fit parameters:')
            print(f'amplitude: {popt[0]:.3f}  phase: {popt[2]:.3f}  constant: {popt[1]:.3f}')
            plt.plot(np.linspace(0,2,101),f_sin0(np.linspace(0,2,101),*popt),c='tab:blue',linewidth=1,alpha=0.8)
            
            tbl = Table(names=['sin_fit_amplitude','sin_fit_phase','sin_fit_constant','sin_fit_type'],dtype=['f8','f8','f8','S99'])
            tbl.add_row([popt[0],popt[2],popt[1],'amplitude*np.sin(2*np.pi*(x+phase))+constant'])
            tbl.write(save_name.replace('.png','_sinparas.csv'),format='ascii.csv',overwrite=True)
            print('>> Sin fit parameters have been saved to', save_name.replace('.png','_sinparas.csv'))
    
    plt.xlabel('Phase',size=12)
    plt.ylabel(col_name,size=12)
    plt.xticks(np.arange(0,2.1,0.2),size=12)
    plt.yticks(size=12)
    plt.title(f'Data points: {len(ts)}  Bin points: {bin_size}  Bin time: {period.to(u.hr).value/bin_size:.3f} hr  Amplitude: '+str(np.round(amp,4)),\
              fontsize=7)
    
    tbl = Table(names=['folded_period(day)','epoch_time(JD)','semi-amplitude','aggregate_func'],\
                dtype=['f8','f8','f8','S65'])
    tbl.add_row([period.to(u.day),epoch_time.jd,amp,str(func)])
    tbl.write(save_name.replace('.png','.csv'),format='ascii.csv',overwrite=True)
    print('>> Folded results have been saved to', save_name.replace('.png','.csv'))
    
    if invert_y:
        ax = plt.gca()
        ax.invert_yaxis()
        plt.text(1.37, min(ts_folded[col_name])+0.05*(max(ts_folded[col_name])-min(ts_folded[col_name])),
                 '$P$='+f'{period.to(u.day).value:.6f} d',size=10.5)
    else:
        plt.text(1.37, max(ts_folded[col_name])-0.05*(max(ts_folded[col_name])-min(ts_folded[col_name])),
                 '$P$='+f'{period.to(u.day).value:.6f} d',size=10.5)
    if save_name is not None:
        plt.savefig(save_name, dpi=240, bbox_inches = 'tight')
    if ifshow:
        plt.show()
    plt.close(f)
    plt.close('all')
    return (period, epoch_time, ts_folded_phase+0.5, ts_folded[col_name], binned_phase, binned_data)

# tbrvs = Table.read('./RUN/20240625_test2/000000_J061149.08+224932.7_LAMOST-LB1/data/sp/'+'LMDR10LRS_rvs_bjd.csv')
# rvbjd = Time(tbrvs['time'], format='jd', scale='tdb')
# rvs = tbrvs['rv']
# rvs_err = tbrvs['rv_err']
# ts_rv = TimeSeries(time=rvbjd, data={"rv":rvs, "rv_err":rvs_err})
# print('RV curves have been generated. Data points:',len(ts_rv))
# print('>>> LSP short for RV curve:')
# tmin = 2.0*u.day
# tmax = 800*u.day
# samples_factor = 5
# #lsp_data = cal_LSP_from_ts(ts=ts_rv, col_name='rv', tmin=tmin, tmax=tmax, samples_factor=samples_factor, level_select=3,ifshow=True,
# #                                      save_name='./Temp/LSP2_lamost_rv_curve_bjd.png')    
# foldeddata = fold_ts(ts_rv, col_name='rv', period=lsp_data[0], epoch_time=None, bin_size=15, func=np.nanmean, ifshow=True,
#                                 save_name='./Temp/LSP2_lamost_rv_curve_bjd_folded.png')

In [13]:
def find_peaks(frequency,power, height, distance, Alarm_level,probability, maxn=50, save_path='./find_peaks.png', plot_log=False, ifshow=False):
    print('>>> Find all peaks....')
    print('height',height,'   distance',distance)
    peaks, properties = scipy.signal.find_peaks(power, height=height, distance=distance, threshold=None)

    if len(peaks)>maxn:
        print(f'Peak number too many (>{maxn}). Reset distance to 5 times of distance = {distance}.')
        peaks, properties = scipy.signal.find_peaks(power, height=height, distance=distance*5, threshold=None)
        if len(peaks)>maxn:
            print(f'Peak number still too many (>{maxn}). Reset distance to 10 times of distance = {distance}.')
            peaks, properties = scipy.signal.find_peaks(power, height=height, distance=distance*10, threshold=None)
            if len(peaks)>maxn:
                print(f'Peak number still too many (>{maxn}). Reset height to 1.8 times of height = {height:.4f}.')
                peaks, properties = scipy.signal.find_peaks(power, height=height*1.8, distance=distance*10, threshold=None)
            
    # 按峰高度排序
    height_index = np.argsort(properties['peak_heights'])
    height_index = np.flipud(height_index)

    print('>>> Find result:\nfind peaks:', len(peaks),
          '\npeak index:', peaks[height_index], 
          '\npeak heights:', properties['peak_heights'][height_index],
          '\npeak frequencies:', frequency[peaks[height_index]],
          '\npeak periods:', 1/frequency[peaks[height_index]])
    
    if save_path is not None:
        f = plt.figure(figsize=(6,4), dpi=200)
        plt.plot(frequency, power, linewidth=.6)   
        plt.plot(frequency[peaks], properties['peak_heights'], color='red', marker='*',markersize=5, linewidth=0)

        for i in range(len(peaks)):
            plt.text(frequency[peaks[i]].value+0.0012,
                     properties['peak_heights'][i] + max(power) * 0.01,
                     str(round(1/frequency[peaks[i]].value,6))+' d',
                     size=4.8)
        plt.xlabel(r'Frequency / $\rm d^{-1}$',size=12)
        plt.ylabel('Amplitude',size=12)
        yy = 0.008
        if plot_log:
            plt.yscale('log')
            yy = 0.001
        plt.axhline(y=Alarm_level, ls=":", c="black", linewidth=0.7)
        xtext = max(frequency) - 0.10 * (max(frequency)-min(frequency))
        plt.text(xtext.value, Alarm_level + 0.008 * max(power),
                 'FAL: '+str(probability), fontsize=6.5)
        plt.title(f'Peak Number: {len(peaks)}  height: {height:.4f}  distance: {distance}',fontsize=9)
        print()
        plt.savefig(save_path, dpi=200, bbox_inches = 'tight')
        if ifshow:
            plt.show()
        plt.close(f)
        plt.close('all')
    
    return (peaks[height_index], properties['peak_heights'][height_index], 
            frequency[peaks[height_index]], 1/frequency[peaks[height_index]])

In [24]:
def analyze_tess(tesslc, lcname, final_path, tmin=24, tmax=15, samples_factor=30,window_length=991,break_tolerance=5,peaklim=9,var_limit=0.01,\
                 smooth_factor=0.00022, downsample_bin_time=None, estimate_sigma=6.0,cut_lo=0.01,cut_hi=0.02):
    # tesslc是光变曲线文件的路径
    # lcname是原始光变曲线的名称或者需要保存的名称前缀
    # final_path是保存分析结果的路径
    # period, best_frequency, frequency, power, probabilities, Alarm_level, lc_flatten, lc_trend, LSperiodogram

    # ww = int(1*18*3600/exptime)+500 # 大于一天的变化用拟合去除   *分析长时标采用
    # ww = int(1*24*3600/exptime)     # 大于一天的变化用拟合去除 *分析短时标采用
    tmin = tmin # 分钟
    tmax = tmax # 天
    ww = window_length
    wl = ww if ww%2==1 else ww+1 # 用于原始光变曲线预展平的window_length
    wl2 = 9991 # prewhitens用于时展平的window_length
    sp = int((tmax*samples_factor)//2*2+1) # sample_peak
    nb1 = 15 # 折叠时的bin数量
    nb2 = 27 # 确定极小值时刻的bin数量
    smooth_factor = smooth_factor # 确定极小值时刻的平滑因子
    sstd = 5 # 折叠时显示的std范围
    pb = [0.005,0.001,0.0005,0.0001,0.00001] # 概率
    peaklim = peaklim # 限制折叠峰值数量
    
    print('flatten window length:',wl)
    print('LSP sample per peak:', sp)
    print('FAL threashold:',pb[1])
    
    print('------ Reading saved TESS lc file.... ------')
    exptime = int(tesslc.split('.csv')[-2].split('_')[-1])
    print('Cadence:',exptime,'s')
    # 1.读取TESS光变曲线，得到(btjd,bjd,flux,fluxerr)
    if 'SPOC' in tesslc:
        datatess = read_tess(tesslc)
    # elif 'QLP' in tesslc:
    #     datatess = read_tess_QLP(tesslc)
        
    # 2.生成光变曲线对象
    print('------ Generating TESS lc.... ------')
    lc_btjd, lc_jd = gen_tess_lc(datatess, sigma=99)
    # 降采样
    if downsample_bin_time is not None:
        print('Aggregate Downsampling: time_bin_size =',downsample_bin_time)
        ts_binned = aggregate_downsample(lc_jd, time_bin_size=downsample_bin_time, aggregate_func=np.nanmean)
        sjd = Time(ts_binned.time_bin_start.jd, format='jd', scale='tdb')
        lc_jd = lk.LightCurve(time = sjd, flux = ts_binned['flux'], flux_err = ts_binned['flux_err'])
        print('Downsampled lc (time=time_bin_start) has been generated. Data points:',len(lc_jd))
    if len(lc_jd['flux'])>0:
        # 先判断光变显不显著
        cp_sigma = estimate_sigma
        print('Calculating variability....')
        print(' > clip sigma ratio:',cp_sigma)
        flux_clip    = sigma_clipping.sigma_clip(lc_jd['flux'],    sigma=cp_sigma, maxiters=1, cenfunc='median', stdfunc='std', masked=False)
        fluxerr_clip = sigma_clipping.sigma_clip(lc_jd['flux_err'],sigma=cp_sigma, maxiters=1, cenfunc='median', stdfunc='std', masked=False)
        medianflux = np.nanmedian(flux_clip)
        maxflux = np.nanmax(flux_clip)
        minflux = np.nanmin(flux_clip)
        deltaflux = maxflux-minflux
        vamp = 0.5*deltaflux/medianflux # 半振幅除以中位流量得到流量变化率
        print(' —> flux variation (relative semi-amplitude):', np.round(vamp*100,2),'%')
        # 计算变化性
        meanflux = np.nanmean(flux_clip)
        stdflux = np.nanstd(flux_clip)
        mean_fluxerr = np.nanmean(fluxerr_clip)
        median_fluxerr = np.nanmedian(fluxerr_clip)
        std_meanerr_ratio = stdflux/mean_fluxerr  # 标准差除以平均测量误差
        std_medianerr_ratio = stdflux/median_fluxerr  # 标准差除以中位测量误差
        print(f'  > flux - median: {medianflux:.1f}  mean: {meanflux:.1f}  delta: {deltaflux:.1f}  std: {stdflux:.1f}  '+\
              f'relative_semi-amplitude: {vamp*100:.2f} %\n'+\
              f'    median_fluxerr: {median_fluxerr:.1f}  std/medianerr: {std_medianerr_ratio:.2f}  '+\
              f'mean_fluxerr: {mean_fluxerr:.1f}  std/meanerr: {std_meanerr_ratio:.2f}')
        tbl = Table(names=['flux_points','clipped_flux_points',
                           'median_flux','mean_flux','min_flux','max_flux','delta_flux','semi_amplitude_medianflux_ratio','std_flux',
                           'mean_flux_err','std_meanerr_ratio','median_flux_err','std_medianerr_ratio','clip_sigma'],\
                   dtype=['i4','i4','f4','f4','f4','f4','f4','f4','f4','f4','f4','f4','f4','f4'])
        tbl.add_row([len(lc_jd['flux']),len(flux_clip),
                     medianflux,meanflux,minflux,maxflux,deltaflux,vamp,stdflux,
                     mean_fluxerr,std_meanerr_ratio,median_fluxerr,std_medianerr_ratio,cp_sigma])
        tbl.write(final_path+f'{lcname}_S00_statistics.csv',format='ascii.csv',overwrite=True)        
        
        
        if vamp>var_limit: # 光变大于1%
            print(f' —> —> Relative semi-amplitude larger than {var_limit*100:.2f}%, starting analysis....')
            # 3.计算周期图
            print('>>>> S01: Calculating LSP....')
            lspdata = cal_LombScargle_fromlc(lc_jd, window_length=wl, break_tolerance=break_tolerance,cut_lo=cut_lo,cut_hi=cut_hi,
                                    tmin=tmin*u.min, tmax=tmax*u.day, sample_peak=sp, probabilities=pb, # 默认绘制第二个概率
                                    save_path=final_path + f'{lcname}_S01_LSP.png')
            lsp0 = lspdata
            # 4.绘制正弦模型
            ndata = len(lspdata[6].time)
            if ndata>90000:
                time_lim = (lspdata[6].time.jd[0], lspdata[6].time.jd[6000])
            elif ndata>10000:
                time_lim = (lspdata[6].time.jd[0], lspdata[6].time.jd[3300])
            elif ndata>4000:
                time_lim = (lspdata[6].time.jd[0], lspdata[6].time.jd[2000])
            elif ndata>1500:
                time_lim = (lspdata[6].time.jd[0], lspdata[6].time.jd[1000])
            elif ndata>600:
                time_lim = (lspdata[6].time.jd[0], lspdata[6].time.jd[400])
            else:
                time_lim = (lspdata[6].time.jd[0], lspdata[6].time.jd[-1])
            print('>>>> S02: Ploting LS model....')
            modeldata = plot_LSmodel(lspdata, xlim=time_lim, sigma_std=sstd, 
                                                save_path=final_path + f'{lcname}_S02_BestModel.png')
            model_time1 = modeldata[0].time.jd
            model_flux1 = modeldata[1]
            # 5.按照给定峰值折叠光变曲线
            print('>>>> S03: Fold lc with best period....')
            foldeddata = fold_lc_fromLS(lspdata, save_path=final_path + f'{lcname}_S03_BestModelFold.png',
                                        nbin=nb1, nstd=sstd, mode='median', errorbar=True)

            # 6.确定极小值时刻
            print('>>>> S04: Determine minimum epoch....')
            try:
                epochfolddata = determine_epoch(lspdata,nbin=nb2,nbin2=nb1,smooth_factor=smooth_factor,
                                                           save_path=final_path + f'{lcname}_S04_PhaseProfile.png')
                period_e, min_phasee, epoch_time0, time_shift, new_epoch, newfolddata, min_times = epochfolddata
                print('>>> Plot minimum times....')
                fig, ax=plt.subplots(figsize=(24,4),dpi=300)
                ax = lspdata[6].plot(ax=ax,color='k',linewidth=0.36)
                ax.plot(modeldata[0].time.jd, modeldata[1], linewidth=0.5, marker='o', markersize=0.4, c='tab:blue',mec=None)
                ax.set_title(lcname+f'  JD(min) = {new_epoch} + {period_e.value} * E',fontsize=9)
                ax.set_xlim(time_lim)
                for i in range(100):
                    xjd = min_times[i]
                    if xjd < time_lim[-1]:
                        ax.axvline(x=xjd, ls='--',linewidth=0.8)
                fig.savefig(final_path + f'{lcname}_S04_MinTimes.png', bbox_inches = 'tight', dpi=300)
                plt.close(fig)
                plt.close('all')
                
                tb1 = Table(names=['epoch_time(JD)','period(day)','E','min_time(JD)'],dtype=['f8','f8','i4','f8'])
                for i in range(len(min_times)):
                    tb1.add_row([new_epoch,period_e.value,i,min_times[i]])
                tb1.write(final_path + f'{lcname}_S04_MinTimes.csv',format='ascii.csv',overwrite=True)
                print('pridicted min times have been saved to', final_path + f'{lcname}_S04_MinTimes.csv')
            except:
                print('Failed to determin epoch.')
                
            if max(lspdata[3])>lspdata[5][1]:
                # 7.找所有峰值
                print('>>>> S05: Find peaks in LSP....')
                peakdata = find_peaks(frequency=lspdata[2], power=lspdata[3], 
                                                 height=lspdata[5][1], distance=sp*2.5, # 用第二个概率作为阈值
                                                 Alarm_level=lspdata[5][1], probability=lspdata[4][1], 
                                                 save_path=final_path + f'{lcname}_S05_FindPeaks.png', plot_log=True)
                peaks, peak_heights, peak_frequencies, peak_periods = peakdata
                fal_threashold = lspdata[5][1]
                fal_threasholds = [fal_threashold for i in range(len(peaks))]
                fal_prob = lspdata[4][1]
                fal_probs = [fal_prob for i in range(len(peaks))]
                # 8.折叠所有峰值
                print('>>>> S06: Fold lc with all peaks...')
                binned_amplitudes = []
                for j in range(len(peaks)):
                    if j < peaklim:
                        print(f' --> fold peak {j}:')
                        foldeddata = fold_ts(lspdata[6], 'flux', period=peak_periods[j], 
                                                        bin_size=nb1, 
                                                        func=np.nanmean,
                                                        dot=True,
                                                        save_name=final_path + f'{lcname}'+'_S06_FoldPeak[%04d].png'%(j))
                        lc_folded_binned_flux = foldeddata[-2]
                        binned_amplitude = (np.nanmax(lc_folded_binned_flux)-np.nanmin(lc_folded_binned_flux))/2
                        binned_amplitudes.append(binned_amplitude)
                    else:
                        binned_amplitudes.append(np.nan)
                if len(peaks)>0:
                    print('>>> Save all peaks to csv file...')
                    peak_data_save = {'peak_height':peak_heights,'peak_frequency':peak_frequencies,'peak_period':peak_periods,\
                                      'binned_amplitude':binned_amplitudes,\
                                      'fal_threashold':fal_threasholds,'fal_probability':fal_probs}
                    df = pd.DataFrame(peak_data_save)
                    df.to_csv(final_path + f'{lcname}_S06_LSP_peak_list.csv', index=True)
                    print('peak list has been saved to', final_path + f'{lcname}_zg_S06_LSP_peak_list.csv')

                # 重复运行该cell即可连续进行prewhiten操作
                print('>>>> S07: PREWHITEN PROCESS 1 ....')
                # 8.预白化1，剔除最大峰值
                lc_resi1 = prewhiten(modeldata, save_path=final_path + f'{lcname}_S07_Prewhiten1_lc.png')
                # 3.计算周期图
                lspdata = cal_LombScargle_fromlc(lc_resi1, window_length=wl2, break_tolerance=break_tolerance,cut_lo=cut_lo,cut_hi=cut_hi,
                                                            tmin=tmin*u.min, tmax=tmax*u.day, sample_peak=sp, probabilities=pb,
                                                            save_path=final_path + f'{lcname}_S07_Prewhiten1_LSP.png')
                # 4.绘制正弦模型
                modeldata = plot_LSmodel(lspdata, xlim=time_lim, sigma_std=sstd, 
                                                    save_path=final_path + f'{lcname}_S07_Prewhiten1_BestModel.png')
                model_time2 = modeldata[0].time.jd
                model_flux2 = modeldata[1]

                # 5.按照给定峰值折叠光变曲线
                foldeddata = fold_lc_fromLS(lspdata, save_path=final_path + f'{lcname}_S07_Prewhiten1_BestModelFold.png',
                                                       nbin=nb1, nstd=sstd, mode='median', errorbar=True)
                # 绘制两个成分叠加
                fig, ax=plt.subplots(figsize=(16,4),dpi=300)
                ax = lsp0[6].plot(ax=ax,color='k',linewidth=0.4)
                ax.plot(model_time1, model_flux1+model_flux2-1, linewidth=0.6, marker='o', markersize=0.6, c='tab:orange')
                ax.set_title(f'Two component model  P1={lsp0[0].value:.7f} d  P2={lspdata[0].value:.7f} d', fontsize=9)
                ax.set_xlim(time_lim)
                fig.savefig(final_path + f'{lcname}_S07_Prewhiten1_TwoCompoModel.png', bbox_inches = 'tight', dpi=300)    
                plt.close()
                plt.close('all')
                # 6.确定极小值时刻
                print('>>>> S08: Prewhiten1 - Determine epoch....')
                try:
                    epochfolddata = determine_epoch(lspdata,nbin=nb2,nbin2=nb1,smooth_factor=smooth_factor,
                                                               save_path=final_path + f'{lcname}_S08_Prewhiten1_PhaseProfile.png')
                    period_e, min_phasee, epoch_time0, time_shift, new_epoch, newfolddata, min_times = epochfolddata
                    fig, ax=plt.subplots(figsize=(24,4),dpi=300)
                    ax = lspdata[6].plot(ax=ax,color='k',linewidth=0.36)
                    ax.plot(modeldata[0].time.jd, modeldata[1], linewidth=0.5, marker='o', markersize=0.4,c='tab:blue',mec=None)
                    ax.set_title(lcname+f'  JD(min) = {new_epoch} + {period_e.value} * E',fontsize=9)
                    ax.set_xlim(time_lim)
                    for i in range(100):
                        xjd = min_times[i]
                        if xjd < time_lim[-1]:
                            ax.axvline(x=xjd, ls='--',linewidth=0.8)
                    fig.savefig(final_path + f'{lcname}_S08_Prewhiten1_MinTimes.png', bbox_inches = 'tight', dpi=300)
                    plt.close(fig)
                    plt.close('all')
    
                    tb1 = Table(names=['epoch_time(JD)','period(day)','E','min_time(JD)'],dtype=['f8','f8','i4','f8'])
                    for i in range(len(min_times)):
                        tb1.add_row([new_epoch,period_e.value,i,min_times[i]])
                    tb1.write(final_path + f'{lcname}_S08_Prewhiten1_MinTimes.csv',format='ascii.csv',overwrite=True)
                    print('pridicted min times have been saved to', final_path + f'{lcname}_S08_MinTimes.csv')
                except:
                    print('Failed to determin epoch.')


                print('>>>> S09: PREWHITEN PROCESS 2 ....')
                # 8.预白化2，剔除最大峰值
                lc_resi2 = prewhiten(modeldata, save_path=final_path + f'{lcname}_S09_Prewhiten2_lc.png')
                # 3.计算周期图
                lspdata = cal_LombScargle_fromlc(lc_resi2, window_length=wl2,break_tolerance=break_tolerance,cut_lo=cut_lo,cut_hi=cut_hi,\
                                                            tmin=tmin*u.min, tmax=tmax*u.day,\
                                                            sample_peak=sp, probabilities=pb,\
                                                            save_path=final_path + f'{lcname}_S09_Prewhiten2_LSP.png')
                # 4.绘制正弦模型
                modeldata = plot_LSmodel(lspdata, xlim=time_lim, sigma_std=sstd, 
                                                    save_path=final_path + f'{lcname}_S09_Prewhiten2_BestModel.png')
                # 5.按照给定峰值折叠光变曲线
                foldeddata = fold_lc_fromLS(lspdata, save_path=final_path + f'{lcname}_S09_Prewhiten2_BestModelFold.png',\
                                                       nbin=9, nstd=5, mode='median', errorbar=True)
        else:
            print(f' —> No significant light variation. (* amplitude < {var_limit*100:.2f}%)')
    else:
        print('lc data length is 0.')
        
# analyze_tess('./RUN/20240626_test1/000007_J065237.19+243622.2_LAMOST J065237.19+243622.1/data/lc/_J065237.19+243622.2_06_TESS Sector 71_SPOC_120.csv',
#               'test5', './Temp/lc_ana_test/',samples_factor=3,downsample_bin_time=10*u.min)
# analyze_tess('./RUN/20240626_test1/000007_J065237.19+243622.2_LAMOST J065237.19+243622.1/data/lc/_J065237.19+243622.2_06_TESS Sector 71_SPOC_120.csv',
#              'test6', './Temp/lc_ana_test/',samples_factor=3)

In [28]:
# 更改参数用于搜寻最短的周期
def analyze_ztf(ztflc, lcname, final_path, min_data=100,tmin=4,tmax=1/24,sp=5,window_length=201,break_tolerance=5,peaklim=9,\
                smooth_factor=0.00021,minamp=1.02,cut_lo=0.016,cut_hi=0.02,clip_sigma_ratio=50):
    # ztflc是光变曲线文件的路径
    # lcname是原始光变曲线的名称或者需要保存的名称前缀
    # final_path是保存分析结果的路径
    # period, best_frequency, frequency, power, probabilities, Alarm_level, lc_flatten, lc_trend, LSperiodogram
    returns = []
    
    ww = window_length # 用于原始光变曲线预展平的初始值window_length
    tmin = tmin # 分钟
    tmax = tmax # 天
    wl = ww if ww%2==1 else ww+1 # 用于原始光变曲线预展平的window_length
    wl2 = 9991 # prewhitens用于时展平的window_length
    sp = sp # sample_peak
    nb1 = 11 # 折叠时的bin数量
    nb2 = 25 # 确定极小值时刻的bin数量
    smooth_factor = smooth_factor # 确定极小值时刻的平滑因子
    sstd = 5 # 折叠时显示的std范围
    pb = [0.01,0.005,0.001,0.0001] # 概率，默认绘制第二个
    peaklim = peaklim # 限制折叠峰值数量    
    minamp = minamp # ztf星等变化阈值
    
    print('flatten window length:',wl)
    print('LSP sample per peak:', sp)
    print('FAL threashold:',pb[1])
    
    print('------ Reading saved ZTF lc file.... ------')
    # 1.读取ZTF光变曲线
    dataztf = read_ztf(ztflc) # HJD,mag,magerr
        
    # 2.生成光变曲线对象
    print('------ Generating ZTF lc.... ------')
    lczgm, lczrm, lczim = gen_ztf_lc_mag(dataztf)
    lczg, lczr, lczi = gen_ztf_lc(dataztf, sigma=clip_sigma_ratio)
    # print(lczr, lczrm)
    
    nzg, nzr, nzi = len(lczg.time),len(lczr.time),len(lczi.time)
    print(f'zg {nzg}\tzr {nzr}\tzi {nzi}')

    data_threashold = min_data # 至少多少个数据点才用于分析
    print('-> minimum data points for analysis:', data_threashold)
    analist = []
    if nzg>data_threashold:
        analist.append('zg')
    if nzr>data_threashold:
        analist.append('zr')    
    if nzi>data_threashold:
        analist.append('zi')
    if len(analist)>0:
        anatext = ''
        for stri in analist:
            anatext += (stri+', ')
        anatext = anatext[:-2]
        print('-> will analyze:',anatext)

    # 统计光变幅度
    print('------ Estimating variation significance.... ------')
    cp_sigma = 8.0
    print('sigma clip ratio:',cp_sigma)
    # zg
    if nzg>data_threashold: # 可以改成0或data_threashold
        mags_clip = sigma_clipping.sigma_clip(lczgm['mag_zg'],sigma=cp_sigma, maxiters=1, cenfunc='median', stdfunc='std', masked=False)
        magerrs_clip = sigma_clipping.sigma_clip(lczgm['mag_err'],sigma=cp_sigma, maxiters=1, cenfunc='median', stdfunc='std', masked=False)
        mags_clip_mean = np.nanmean(mags_clip)
        mags_clip_std = np.nanstd(mags_clip)             # 计算星等标准差
        magerrs_clip_mean = np.nanmean(magerrs_clip)     # 计算平均测光星等误差
        gmag_std_ratio = mags_clip_std/magerrs_clip_mean  # 标准差除以平均测量误差
        print(f'[zg]  mag_mean:{mags_clip_mean:.3f}  mag_std:{mags_clip_std:.4f}  mean_magerr:{magerrs_clip_mean:.4f}'+\
              f'  mag_std/mean_magerr:{gmag_std_ratio:.3f}')
        tbl = Table(names=['zg_data_points','clipped_zg_data_points',
                           'zg_mags_min','zg_mags_max','delta_zg','zg_mags_median','zg_mags_mean','zg_mags_std',
                           'zg_max_magerr','zg_median_magerr','zg_mean_magerr','zg_std_magerr_ratio','clip_sigma'],\
                   dtype=['i4','i4','f4','f4','f4','f4','f4','f4','f4','f4','f4','f4','f4'])
        tbl.add_row([nzg,len(mags_clip),np.nanmin(mags_clip),np.nanmax(mags_clip),np.nanmax(mags_clip)-np.nanmin(mags_clip),
                     np.nanmedian(mags_clip),mags_clip_mean,mags_clip_std,
                     np.nanmax(magerrs_clip),np.nanmedian(magerrs_clip),magerrs_clip_mean,gmag_std_ratio,cp_sigma])
        tbl.write(final_path + f'{lcname}_zg_statistics.csv',format='ascii.csv',overwrite=True)
    if nzr>data_threashold: # 可以改成0或data_threashold
        mags_clip = sigma_clipping.sigma_clip(lczrm['mag_zr'],sigma=cp_sigma, maxiters=1, cenfunc='median', stdfunc='std', masked=False)
        magerrs_clip = sigma_clipping.sigma_clip(lczrm['mag_err'],sigma=cp_sigma, maxiters=1, cenfunc='median', stdfunc='std', masked=False)
        mags_clip_mean = np.nanmean(mags_clip)
        mags_clip_std = np.nanstd(mags_clip)             # 计算星等标准差
        magerrs_clip_mean = np.nanmean(magerrs_clip)     # 计算平均测光星等误差
        rmag_std_ratio = mags_clip_std/magerrs_clip_mean  # 标准差除以平均测量误差
        print(f'[zr]  mag_mean:{mags_clip_mean:.3f}  mag_std:{mags_clip_std:.4f}  mean_magerr:{magerrs_clip_mean:.4f}'+\
              f'  mag_std/mean_magerr:{rmag_std_ratio:.3f}')
        tbl = Table(names=['zr_data_points','clipped_zr_data_points',
                           'zr_mags_min','zr_mags_max','delta_zr','zr_mags_median','zr_mags_mean','zr_mags_std',
                           'zr_max_magerr','zr_median_magerr','zr_mean_magerr','zr_std_magerr_ratio','clip_sigma'],\
                   dtype=['i4','i4','f4','f4','f4','f4','f4','f4','f4','f4','f4','f4','f4'])
        tbl.add_row([nzr,len(mags_clip),np.nanmin(mags_clip),np.nanmax(mags_clip),np.nanmax(mags_clip)-np.nanmin(mags_clip),
                     np.nanmedian(mags_clip),mags_clip_mean,mags_clip_std,
                     np.nanmax(magerrs_clip),np.nanmedian(magerrs_clip),magerrs_clip_mean,rmag_std_ratio,cp_sigma])
        tbl.write(final_path + f'{lcname}_zr_statistics.csv',format='ascii.csv',overwrite=True)    
    if nzi>10: # 可以改成0或data_threashold
        print('zi number > 10:')
        mags_clip = sigma_clipping.sigma_clip(lczim['mag_zi'],sigma=cp_sigma, maxiters=1, cenfunc='median', stdfunc='std', masked=False)
        magerrs_clip = sigma_clipping.sigma_clip(lczim['mag_err'],sigma=cp_sigma, maxiters=1, cenfunc='median', stdfunc='std', masked=False)
        mags_clip_mean = np.nanmean(mags_clip)
        mags_clip_std = np.nanstd(mags_clip)             # 计算星等标准差
        magerrs_clip_mean = np.nanmean(magerrs_clip)     # 计算平均测光星等误差
        imag_std_ratio = mags_clip_std/magerrs_clip_mean  # 标准差除以平均测量误差
        print(f'[zi]  mag_mean:{mags_clip_mean:.3f}  mag_std:{mags_clip_std:.4f}  mean_magerr:{magerrs_clip_mean:.4f}'+\
              f'  mag_std/mean_magerr:{imag_std_ratio:.3f}')
        tbl = Table(names=['zi_data_points','clipped_zi_data_points',
                           'zi_mags_min','zi_mags_max','delta_zi','zi_mags_median','zi_mags_mean','zi_mags_std',
                           'zi_max_magerr','zi_median_magerr','zi_mean_magerr','zi_std_magerr_ratio','clip_sigma'],\
                   dtype=['i4','i4','f4','f4','f4','f4','f4','f4','f4','f4','f4','f4','f4'])
        tbl.add_row([nzi,len(mags_clip),np.nanmin(mags_clip),np.nanmax(mags_clip),np.nanmax(mags_clip)-np.nanmin(mags_clip),
                     np.nanmedian(mags_clip),mags_clip_mean,mags_clip_std,
                     np.nanmax(magerrs_clip),np.nanmedian(magerrs_clip),magerrs_clip_mean,imag_std_ratio,cp_sigma])
        tbl.write(final_path + f'{lcname}_zi_statistics.csv',format='ascii.csv',overwrite=True)        

    pg = False
    pr = False
    
    print('-------- Analyzing [zg] lc.... --------')
    lc = lczg
    datapoints = len(lc)
    if datapoints>data_threashold:
        # 判断光变显不显著
        fluxstd = np.nanstd(lc['flux']) # 流量的标准差
        meanerr = np.nanmean(lc['flux_err']) # 流量误差的平均值
        vamp = fluxstd/meanerr
        print(f' —> flux_std: {fluxstd:.3f}   mean_flux_err: {meanerr:.3f}   ratio: {vamp:.3f}')
        if gmag_std_ratio>minamp: # 光变显著
            print(f' —> Variation significant! ratio = {gmag_std_ratio:.3f} > {minamp}')
            # 3.计算周期图
            print('>>>> S01: Calculating LSP....')
            lspdata = cal_LombScargle_fromlc(lc, window_length=wl, break_tolerance=break_tolerance,cut_lo=cut_lo,cut_hi=cut_hi,\
                            tmin=tmin*u.min, tmax=tmax*u.day, sample_peak=sp, probabilities=pb, # 默认绘制第二个概率
                            save_path=final_path + f'{lcname}_zg_S01_LSP.png')
            returns.append(lspdata)
            print('>>> S02: Ploting LS model....')
            # 绘制正弦模型
            ndata = len(lspdata[6].time)
            if ndata>300:
                time_lim = (lspdata[6].time.jd[10], lspdata[6].time.jd[100])
            elif ndata>150:
                time_lim = (lspdata[6].time.jd[10], lspdata[6].time.jd[60])
            else:
                time_lim = (lspdata[6].time.jd[0], lspdata[6].time.jd[-1])
            modeldata = plot_LSmodel(lspdata, xlim=time_lim, sigma_std=sstd, time_step=lspdata[0].value/5,
                                                save_path=final_path + f'{lcname}_zg_S02_BestModel.png')
            pg = lspdata[0].to(u.min).value # 周期
            pgf = lspdata[5][1]      # Alarm_level
            pgl = np.max(lspdata[3]) # max_power
            pgp = lspdata[4][1]      # probabilities
            # 4.按照给定峰值折叠光变曲线
            print('>>>> S03: Fold lc with best period....')
            foldeddata = fold_lc_fromLS(lspdata, save_path=final_path + f'{lcname}_zg_S03_BestModelFold.png',
                                                   nbin=nb1, nstd=sstd, mode='mean', errorbar=True)
            print('>> Fold original lc with magnitudes....')
            foldeddata2 = fold_ts(lczgm, 'mag_zg', period=lspdata[0], bin_size=nb1, func=np.nanmean,invert_y=True,
                                                   save_name=final_path + f'{lcname}_zg_S03_BestModelFold_mag.png')
            # 6.确定极小值时刻
            print('>>>> S04: Determine minimum epoch....')
            try:
                epochfolddata = determine_epoch(lspdata,nbin=nb2,nbin2=nb1,smooth_factor=smooth_factor,
                                                           save_path=final_path + f'{lcname}_zg_S04_PhaseProfile.png')
                period_e, min_phasee, epoch_time0, time_shift, new_epoch, newfolddata, min_times = epochfolddata
                print('>>> Plot minimum times....')
                fig, ax=plt.subplots(figsize=(24,4),dpi=300)
                ax = lspdata[6].plot(ax=ax,color='k',linewidth=0.5)
                ax.plot(modeldata[0].time.jd, modeldata[1], linewidth=0.5, marker='o', markersize=0.6,c='tab:blue',mec=None)
                ax.set_title(f'[zg]  JD(min) = {new_epoch} + {period_e.value} * E',fontsize=9)
                ax.set_xlim(time_lim)
                for i in range(200):
                    xjd = min_times[i]
                    if xjd < time_lim[-1]:
                        ax.axvline(x=xjd, ls='--',linewidth=0.4)
                fig.savefig(final_path + f'{lcname}_zg_S04_MinTimes.png', bbox_inches = 'tight', dpi=300)
                plt.close(fig)
                plt.close('all')
                # 保存星历参数
                tb1 = Table(names=['epoch_time(JD)','period(day)','E','min_time(JD)'],dtype=['f8','f8','i4','f8'])
                for i in range(len(min_times)):
                    tb1.add_row([new_epoch,period_e.value,i,min_times[i]])
                tb1.write(final_path + f'{lcname}_zg_S04_MinTimes.csv',format='ascii.csv',overwrite=True)
                print('pridicted min times have been saved to', final_path + f'{lcname}_zg_S04_MinTimes.csv')
            except:
                print('Failed to determin epoch.')
                
            if max(lspdata[3])>lspdata[5][1]:  # 判断峰值是否高于FAL 
                print(' -> Peak is significant. Proceeding to find more peaks....')
                # 5.找所有峰值
                print('>>>> S05: Find peaks in LSP....')
                peakdata = find_peaks(frequency=lspdata[2], power=lspdata[3], 
                                                 height=lspdata[5][2], distance=sp*15, # *用第3个概率作为阈值
                                                 Alarm_level=lspdata[5][2], probability=lspdata[4][2], 
                                                 save_path=final_path + f'{lcname}_zg_S05_FindPeaks.png', plot_log=False)
                peaks, peak_heights, peak_frequencies, peak_periods = peakdata
                fal_threashold = lspdata[5][2]
                fal_threasholds = [fal_threashold for i in range(len(peaks))]
                fal_prob = lspdata[4][2]
                fal_probs = [fal_prob for i in range(len(peaks))]
                # 6.折叠所有峰值
                print('>>>> S06: Fold lc with all peaks...')
                binned_amplitudes = []
                for j in range(len(peaks)):
                    if j < peaklim:
                        print(f' --> fold peak {j}:')
                        foldeddata = fold_ts(lspdata[6], 'flux', period=peak_periods[j], 
                                                        bin_size=nb1, 
                                                        func=np.nanmean,
                                                        save_name=final_path + f'{lcname}'+'_zg_S06_FoldPeak[%02d].png'%(j))
                        lc_folded_binned_flux = foldeddata[-2]
                        binned_amplitude = (np.nanmax(lc_folded_binned_flux)-np.nanmin(lc_folded_binned_flux))/2
                        binned_amplitudes.append(binned_amplitude)
                    else:
                        binned_amplitudes.append(np.nan)
                if len(peaks)>0:
                    print('>>> Save all peaks to csv file...')
                    peak_data_save = {'peak_height':peak_heights,'peak_frequency':peak_frequencies,'peak_period':peak_periods,\
                                      'binned_amplitude':binned_amplitudes,\
                                      'fal_threashold':fal_threasholds,'fal_probability':fal_probs}
                    df = pd.DataFrame(peak_data_save)
                    df.to_csv(final_path + f'{lcname}_zg_S06_LSP_peak_list.csv', index=True)
                    print('peak list has been saved to', final_path + f'{lcname}_zg_S06_LSP_peak_list.csv')
                if len(peaks)>1:            
                    # 重复运行该cell即可连续进行prewhiten操作
                    print('>>>> S07: PREWHITEN PROCESS 1 ....')
                    # 7.预白化1，剔除最大峰值
                    lc_resi1 = prewhiten(modeldata, save_path=final_path + f'{lcname}_zg_S07_Prewhiten1_lc.png')
                    # 8.计算周期图
                    lspdata = cal_LombScargle_fromlc(lc_resi1, window_length=wl2, break_tolerance=break_tolerance,cut_lo=cut_lo,cut_hi=cut_hi,\
                                                                tmin=tmin*u.min, tmax=tmax*u.day, sample_peak=sp, probabilities=pb,
                                                                save_path=final_path + f'{lcname}_zg_S07_Prewhiten1_LSP.png')
                    # 9.按照给定峰值折叠光变曲线
                    foldeddata = fold_lc_fromLS(lspdata, save_path=final_path+f'{lcname}_zg_S07_Prewhiten1_BestModelFold.png',
                                                         nbin=nb1, nstd=sstd, mode='mean', errorbar=True)
            else:
                print('Peak is not significant. Analysis ended.')
        else:
            print(' —> No significant light variation.')
    else:
        print('Data points to few.')
        
    print('-------- Analyzing [zr] lc.... --------')
    lc = lczr
    datapoints = len(lc)
    if datapoints>data_threashold:
        # 判断光变显不显著
        fluxstd = np.nanstd(lc['flux']) # 流量的标准差
        meanerr = np.nanmean(lc['flux_err']) # 流量误差的平均值
        vamp = fluxstd/meanerr
        print(f' —> flux_std: {fluxstd:.3f}   mean_flux_err: {meanerr:.3f}   ratio: {vamp:.3f}')
        if rmag_std_ratio>minamp: # 光变显著
            print(f' —> Variation significant! ratio = {rmag_std_ratio:.3f} > {minamp}')
            # 3.计算周期图
            print('>>>> S01: Calculating LSP....')
            lspdata = cal_LombScargle_fromlc(lc, window_length=wl, break_tolerance=break_tolerance,cut_lo=cut_lo,cut_hi=cut_hi,\
                            tmin=tmin*u.min, tmax=tmax*u.day, sample_peak=sp, probabilities=pb, # 默认绘制第二个概率
                            save_path=final_path + f'{lcname}_zr_S01_LSP.png')
            returns.append(lspdata)
            print('>>> S02: Ploting LS model....')
            # 绘制正弦模型
            ndata = len(lspdata[6].time)
            if ndata>300:
                time_lim = (lspdata[6].time.jd[10], lspdata[6].time.jd[100])
            elif ndata>150:
                time_lim = (lspdata[6].time.jd[10], lspdata[6].time.jd[60])
            else:
                time_lim = (lspdata[6].time.jd[0], lspdata[6].time.jd[-1])
            modeldata = plot_LSmodel(lspdata, xlim=time_lim, sigma_std=sstd, time_step=lspdata[0].value/5,
                                                save_path=final_path + f'{lcname}_zr_S02_BestModel.png')
            pr = lspdata[0].to(u.min).value # 周期
            prf = lspdata[5][1]      # Alarm_level
            prl = np.max(lspdata[3]) # max_power
            prp = lspdata[4][1]      # probabilities
            # 4.按照给定峰值折叠光变曲线
            print('>>>> S03: Fold lc with best period....')
            foldeddata = fold_lc_fromLS(lspdata, save_path=final_path + f'{lcname}_zr_S03_BestModelFold.png',
                                                   nbin=nb1, nstd=sstd, mode='mean', errorbar=True)
            print('>> Fold original lc with magnitudes....')
            foldeddata2 = fold_ts(lczrm, 'mag_zr', period=lspdata[0], bin_size=nb1, func=np.nanmean,invert_y=True,
                                                   save_name=final_path + f'{lcname}_zr_S03_BestModelFold_mag.png')
            # 6.确定极小值时刻
            print('>>>> S04: Determine minimum epoch....')
            try:
                epochfolddata = determine_epoch(lspdata,nbin=nb2,nbin2=nb1,smooth_factor=smooth_factor,
                                                           save_path=final_path + f'{lcname}_zr_S04_PhaseProfile.png')
                period_e, min_phasee, epoch_time0, time_shift, new_epoch, newfolddata, min_times = epochfolddata
                print('>>> Plot minimum times....')
                fig, ax=plt.subplots(figsize=(24,4),dpi=300)
                ax = lspdata[6].plot(ax=ax,color='k',linewidth=0.5)
                ax.plot(modeldata[0].time.jd, modeldata[1], linewidth=0.5, marker='o', markersize=0.6,c='tab:blue',mec=None)
                ax.set_title(f'[zr]  JD(min) = {new_epoch} + {period_e.value} * E',fontsize=9)
                ax.set_xlim(time_lim)
                for i in range(200):
                    xjd = min_times[i]
                    if xjd < time_lim[-1]:
                        ax.axvline(x=xjd, ls='--',linewidth=0.4)
                fig.savefig(final_path + f'{lcname}_zr_S04_MinTimes.png', bbox_inches = 'tight', dpi=300)
                plt.close(fig)
                plt.close('all')
                # 保存星历参数
                tb1 = Table(names=['epoch_time(JD)','period(day)','E','min_time(JD)'],dtype=['f8','f8','i4','f8'])
                for i in range(len(min_times)):
                    tb1.add_row([new_epoch,period_e.value,i,min_times[i]])
                tb1.write(final_path + f'{lcname}_zr_S04_MinTimes.csv',format='ascii.csv',overwrite=True)
                print('pridicted min times have been saved to', final_path + f'{lcname}_zr_S04_MinTimes.csv')
            except:
                print('Failed to determin epoch.')
                
            if max(lspdata[3])>lspdata[5][1]:  # 判断峰值是否高于FAL 
                print(' -> Peak is significant. Proceeding to find more peaks....')
                # 5.找所有峰值
                print('>>>> S05: Find peaks in LSP....')
                peakdata = find_peaks(frequency=lspdata[2], power=lspdata[3], 
                                                 height=lspdata[5][2], distance=sp*15, # *用第3个概率作为阈值
                                                 Alarm_level=lspdata[5][2], probability=lspdata[4][2], 
                                                 save_path=final_path + f'{lcname}_zr_S05_FindPeaks.png', plot_log=False)
                peaks, peak_heights, peak_frequencies, peak_periods = peakdata
                fal_threashold = lspdata[5][2]
                fal_threasholds = [fal_threashold for i in range(len(peaks))]
                fal_prob = lspdata[4][2]
                fal_probs = [fal_prob for i in range(len(peaks))]
                # 6.折叠所有峰值
                print('>>>> S06: Fold lc with all peaks...')
                binned_amplitudes = []
                for j in range(len(peaks)):
                    if j < peaklim:
                        print(f' --> fold peak {j}:')
                        foldeddata = fold_ts(lspdata[6], 'flux', period=peak_periods[j], 
                                                        bin_size=nb1, 
                                                        func=np.nanmean,
                                                        save_name=final_path + f'{lcname}'+'_zr_S06_FoldPeak[%02d].png'%(j))
                        lc_folded_binned_flux = foldeddata[-2]
                        binned_amplitude = (np.nanmax(lc_folded_binned_flux)-np.nanmin(lc_folded_binned_flux))/2
                        binned_amplitudes.append(binned_amplitude)
                    else:
                        binned_amplitudes.append(np.nan)
                if len(peaks)>0:
                    print('>>> Save all peaks to csv file...')
                    peak_data_save = {'peak_height':peak_heights,'peak_frequency':peak_frequencies,'peak_period':peak_periods,\
                                      'binned_amplitude':binned_amplitudes,\
                                      'fal_threashold':fal_threasholds,'fal_probability':fal_probs}
                    df = pd.DataFrame(peak_data_save)
                    df.to_csv(final_path + f'{lcname}_zr_S06_LSP_peak_list.csv', index=True)
                    print('peak list has been saved to', final_path + f'{lcname}_zr_S06_LSP_peak_list.csv')
                if len(peaks)>1:            
                    # 重复运行该cell即可连续进行prewhiten操作
                    print('>>>> S07: PREWHITEN PROCESS 1 ....')
                    # 7.预白化1，剔除最大峰值
                    lc_resi1 = prewhiten(modeldata, save_path=final_path + f'{lcname}_zr_S07_Prewhiten1_lc.png')
                    # 8.计算周期图
                    lspdata = cal_LombScargle_fromlc(lc_resi1, window_length=wl2, break_tolerance=break_tolerance,cut_lo=cut_lo,cut_hi=cut_hi,\
                                                                tmin=tmin*u.min, tmax=tmax*u.day, sample_peak=sp, probabilities=pb,
                                                                save_path=final_path + f'{lcname}_zr_S07_Prewhiten1_LSP.png')
                    # 9.按照给定峰值折叠光变曲线
                    foldeddata = fold_lc_fromLS(lspdata, save_path=final_path+f'{lcname}_zr_S07_Prewhiten1_BestModelFold.png',
                                                         nbin=nb1, nstd=sstd, mode='mean', errorbar=True)
            else:
                print('Peak is not significant. Analysis ended.')
        else:
            print(' —> No significant light variation.')
    else:
        print('Data points to few.')

    print('>>>> Assessing periods in zg and zr band....')
    if pg and pr:
        diff = abs(pg-pr)/pr # 相对误差
        print(f'period_zg = {pg:.2f} min = {pg/60:.3f} hr\tperiod_zr = {pr:.2f} min = {pr/60:.3f} hr')
        print(f'period difference = {diff*100:.2f} %')
        print(f'zg_FAL_ratio: {pgl/pgf:.3f}\tprob-[{pgp}]    zr_FAL_ratio: {prl/prf:.3f}\tprob-[{prp}]')
        if (diff < 0.05 or pgl > 1.25*pgf or prl > 1.25*prf): # 相差小于5%或者大于FAL 1.25倍，报告检测结果
            if (np.abs(pg-1440)>8.0 or np.abs(pr-1440)>8.0) and (np.abs(pg-720)>7.0 or np.abs(pr-720)>7.0): # 如果至少一个不是24h和12h
                if not (np.abs(pg-1440)<8.0 and np.abs(pr-1440)<8.0): # 排除都是24h
                    if not (np.abs(pg-720)<7.0 and np.abs(pr-720)<7.0): # 排除都是12h
                        if not (pgl < pgf and (np.abs(pr-1440)<8.0 or np.abs(pr-720)<7.0)): # 排除pg不显著并且pr接近12h/24h
                            if not (prl < prf and (np.abs(pg-1440)<8.0 or np.abs(pg-720)<7.0)): # 排除pg不显著并且pr接近12h/24h
                                if not ((np.abs(pg-1440)<8.0 and np.abs(pr-720)<7.0) or (np.abs(pr-1440)<8.0 and np.abs(pg-720)<7.0)):
                                    if diff < 0.05:
                                        ss='_Double-Identification'
                                        print('\n—> Same Period Detected in Both zg and zr Band!')
                                    else:
                                        ss='_Single-Identification'
                                        print('\n—> Significant Period Detected in single band.')
                                    txtdata = ss[1:]+'\n'+\
                                    f'period_zg = {pg:.6f} min = {pg/60:.6f} hr\nperiod_zr = {pr:.6f} min = {pr/60:.6f} hr\ndifference = {diff*100:.3f} %\n'+\
                                    f'period_zg_power = {pgl:.6f}\tperiod_zr_power = {prl:.6f}\n'+\
                                    f'zg_FAL_ratio: {pgl/pgf:.3f}\tprob-[{pgp}]\nzr_FAL_ratio: {prl/prf:.3f}\tprob-[{prp}]'
                                    report_name = final_path+f'{lcname}_Candidate_Period_Report'+ss+".txt"
                                    save_data_totxt(txtdata, save_txt_name=report_name)
                                    print('**************** Candidate Period Report ****************')
                                    #print(txtdata)
                                    print(f'period_zg = {pg:.4f} min\nperiod_zr = {pr:.4f} min\ndifference = {diff*100:.2f} %\n'+\
                                          f'zg_FAL_ratio: {pg/pgf:.3f}\tprob-[{pgp}]\nzr_FAL_ratio: {prl/prf:.3f}\tprob-[{prp}]')
                                    print('*********************************************************')
                                    print(f'>>> Significant Period Report has been saved to {report_name}.\n')

    print('-------- Analyzing [zi] lc.... --------')
    lc = lczi
    datapoints = len(lc)
    if datapoints>data_threashold:
        # 判断光变显不显著
        fluxstd = np.nanstd(lc['flux']) # 流量的标准差
        meanerr = np.nanmean(lc['flux_err']) # 流量误差的平均值
        vamp = fluxstd/meanerr
        print(f' —> flux_std: {fluxstd:.3f}   mean_flux_err: {meanerr:.3f}   ratio: {vamp:.3f}')
        if imag_std_ratio>minamp: # 光变显著
            print(f' —> Variation significant! ratio = {imag_std_ratio:.3f} > {minamp}')
            # 3.计算周期图
            print('>>>> S01: Calculating LSP....')
            lspdata = cal_LombScargle_fromlc(lc, window_length=wl, break_tolerance=break_tolerance,cut_lo=cut_lo,cut_hi=cut_hi,\
                            tmin=tmin*u.min, tmax=tmax*u.day, sample_peak=sp, probabilities=pb, # 默认绘制第二个概率
                            save_path=final_path + f'{lcname}_zi_S01_LSP.png')
            returns.append(lspdata)
            print('>>> S02: Ploting LS model....')
            # 绘制正弦模型
            ndata = len(lspdata[6].time)
            if ndata>300:
                time_lim = (lspdata[6].time.jd[10], lspdata[6].time.jd[100])
            elif ndata>150:
                time_lim = (lspdata[6].time.jd[10], lspdata[6].time.jd[60])
            else:
                time_lim = (lspdata[6].time.jd[0], lspdata[6].time.jd[-1])
            modeldata = plot_LSmodel(lspdata, xlim=time_lim, sigma_std=sstd, time_step=lspdata[0].value/5,
                                                save_path=final_path + f'{lcname}_zi_S02_BestModel.png')
            # 4.按照给定峰值折叠光变曲线
            print('>>>> S03: Fold lc with best period....')
            foldeddata = fold_lc_fromLS(lspdata, save_path=final_path + f'{lcname}_zi_S03_BestModelFold.png',
                                                   nbin=nb1, nstd=sstd, mode='mean', errorbar=True)
            print('>> Fold original lc with magnitudes....')
            foldeddata2 = fold_ts(lczim, 'mag_zi', period=lspdata[0], bin_size=nb1, func=np.nanmean,invert_y=True,
                                                   save_name=final_path + f'{lcname}_zi_S03_BestModelFold_mag.png')
            # 6.确定极小值时刻
            print('>>>> S04: Determine minimum epoch....')
            try:
                epochfolddata = determine_epoch(lspdata,nbin=nb2,nbin2=nb1,smooth_factor=smooth_factor,
                                                           save_path=final_path + f'{lcname}_zi_S04_PhaseProfile.png')
                period_e, min_phasee, epoch_time0, time_shift, new_epoch, newfolddata, min_times = epochfolddata
                print('>>> Plot minimum times....')
                fig, ax=plt.subplots(figsize=(24,4),dpi=300)
                ax = lspdata[6].plot(ax=ax,color='k',linewidth=0.5)
                ax.plot(modeldata[0].time.jd, modeldata[1], linewidth=0.5, marker='o', markersize=0.6,c='tab:blue',mec=None)
                ax.set_title(f'[zi]  JD(min) = {new_epoch} + {period_e.value} * E',fontsize=9)
                ax.set_xlim(time_lim)
                for i in range(200):
                    xjd = min_times[i]
                    if xjd < time_lim[-1]:
                        ax.axvline(x=xjd, ls='--',linewidth=0.4)
                fig.savefig(final_path + f'{lcname}_zi_S04_MinTimes.png', bbox_inches = 'tight', dpi=300)
                plt.close(fig)
                plt.close('all')
                # 保存星历参数
                tb1 = Table(names=['epoch_time(JD)','period(day)','E','min_time(JD)'],dtype=['f8','f8','i4','f8'])
                for i in range(len(min_times)):
                    tb1.add_row([new_epoch,period_e.value,i,min_times[i]])
                tb1.write(final_path + f'{lcname}_zi_S04_MinTimes.csv',format='ascii.csv',overwrite=True)
                print('pridicted min times have been saved to', final_path + f'{lcname}_zi_S04_MinTimes.csv')
            except:
                print('Failed to determin epoch.')
                
            if max(lspdata[3])>lspdata[5][1]:  # 判断峰值是否高于FAL 
                print(' -> Peak is significant. Proceeding to find more peaks....')
                # 5.找所有峰值
                print('>>>> S05: Find peaks in LSP....')
                peakdata = find_peaks(frequency=lspdata[2], power=lspdata[3], 
                                                 height=lspdata[5][2], distance=sp*15, # *用第3个概率作为阈值
                                                 Alarm_level=lspdata[5][2], probability=lspdata[4][2], 
                                                 save_path=final_path + f'{lcname}_zi_S05_FindPeaks.png', plot_log=False)
                peaks, peak_heights, peak_frequencies, peak_periods = peakdata
                fal_threashold = lspdata[5][2]
                fal_threasholds = [fal_threashold for i in range(len(peaks))]
                fal_prob = lspdata[4][2]
                fal_probs = [fal_prob for i in range(len(peaks))]
                # 6.折叠所有峰值
                print('>>>> S06: Fold lc with all peaks...')
                binned_amplitudes = []
                for j in range(len(peaks)):
                    if j < peaklim:
                        print(f' --> fold peak {j}:')
                        foldeddata = fold_ts(lspdata[6], 'flux', period=peak_periods[j], 
                                                        bin_size=nb1, 
                                                        func=np.nanmean,
                                                        save_name=final_path + f'{lcname}'+'_zi_S06_FoldPeak[%02d].png'%(j))
                        lc_folded_binned_flux = foldeddata[-2]
                        binned_amplitude = (np.nanmax(lc_folded_binned_flux)-np.nanmin(lc_folded_binned_flux))/2
                        binned_amplitudes.append(binned_amplitude)
                    else:
                        binned_amplitudes.append(np.nan)
                if len(peaks)>0:
                    print('>>> Save all peaks to csv file...')
                    peak_data_save = {'peak_height':peak_heights,'peak_frequency':peak_frequencies,'peak_period':peak_periods,\
                                      'binned_amplitude':binned_amplitudes,\
                                      'fal_threashold':fal_threasholds,'fal_probability':fal_probs}
                    df = pd.DataFrame(peak_data_save)
                    df.to_csv(final_path + f'{lcname}_zi_S06_LSP_peak_list.csv', index=True)
                    print('peak list has been saved to', final_path + f'{lcname}_zi_S06_LSP_peak_list.csv')
                if len(peaks)>1:            
                    # 重复运行该cell即可连续进行prewhiten操作
                    print('>>>> S07: PREWHITEN PROCESS 1 ....')
                    # 7.预白化1，剔除最大峰值
                    lc_resi1 = prewhiten(modeldata, save_path=final_path + f'{lcname}_zi_S07_Prewhiten1_lc.png')
                    # 8.计算周期图
                    lspdata = cal_LombScargle_fromlc(lc_resi1, window_length=wl2, break_tolerance=break_tolerance,cut_lo=cut_lo,cut_hi=cut_hi,\
                                                                tmin=tmin*u.min, tmax=tmax*u.day, sample_peak=sp, probabilities=pb,
                                                                save_path=final_path + f'{lcname}_zi_S07_Prewhiten1_LSP.png')
                    # 9.按照给定峰值折叠光变曲线
                    foldeddata = fold_lc_fromLS(lspdata, save_path=final_path+f'{lcname}_zi_S07_Prewhiten1_BestModelFold.png',
                                                         nbin=nb1, nstd=sstd, mode='mean', errorbar=True)
            else:
                print(' -> Peak is not significant. Analysis ended.')
        else:
            print(' —> No significant light variation.')
    else:
        print('Data points to few.')
    return returns

# 测试
# analyze_ztf('./RUN/20240627_test6/000003_J201734.00-033951.0_V794 Aql/data/lc/_J201734.00-033951.0_ztf_lc.csv',
#             'test6', 
#             './Temp/ztf_ana_test/', 
#             min_data=10,tmin=80,tmax=1,sp=5,window_length=201,break_tolerance=10)
# analyze_ztf('./RUN/20240627_test6/000013_J025238.57+202958.2_LAMOST-DWD/data/lc/_J025238.57+202958.2_ztf_lc.csv',
#             'test10_dwd', 
#             './Temp/ztf_ana_test/', 
#             min_data=100,tmin=4,tmax=1/180,sp=1,window_length=301,break_tolerance=10,peaklim=9,smooth_factor=0.00021,minamp=1.02)
# analyze_ztf('./RUN_SINGLE/20240724_test2/000000_J025238.57+202958.1_LAMOST-DWD/data/lc/_J025238.57+202958.1_ztf_lc.csv',
#             'test_dwd', 
#             './Temp/ztf_ana_test/', 
#             min_data=100,tmin=4,tmax=1/220,sp=1,window_length=501,break_tolerance=10,peaklim=9,smooth_factor=0.00021,minamp=1.04)

flatten window length: 501
LSP sample per peak: 1
FAL threashold: 0.005
------ Reading saved ZTF lc file.... ------
./RUN_SINGLE/20240724_test2/000000_J025238.57+202958.1_LAMOST-DWD/data/lc/_J025238.57+202958.1_ztf_lc.csv have been read.
data number: zg 392 zr 616 zi 35
------ Generating ZTF lc.... ------
zg 392	zr 616	zi 35
-> minimum data points for analysis: 100
-> will analyze: zg, zr
------ Estimating variation significance.... ------
sigma clip ratio: 8.0
[zg]  mag_mean:16.980  mag_std:0.0207  mean_magerr:0.0198  mag_std/mean_magerr:1.045
[zr]  mag_mean:17.321  mag_std:0.0206  mean_magerr:0.0197  mag_std/mean_magerr:1.044
-------- Analyzing [zg] lc.... --------
 —> flux_std: 30.905   mean_flux_err: 29.422   ratio: 1.050
 —> Variation significant! ratio = 1.045 > 1.04
>>>> S01: Calculating LSP....
flatten lc with window_length = 501   break_tolerance =  10
>> Calculating Lomb-Scargle Periodogram....
LC data point number: 392
Flattened lc with window length: 501
min f 220.0 1 / d 	

>>>> S04: Determine minimum epoch....
lc start time: 2458206.63655976
>>> fold lc with best frequency (epoch is specified to 2458206.63655976)....
lc data points: 392  bin number: 25
Folded Period: 0.0031599 d  0.075837 hr  4.550244 min
binned_amplitude: 0.015266599349738452
epoch_time: 2458206.63655976
original minimum phase: 0.286592
>> corrected Epoch: 2458206.6390453056


>>> fold lc with best frequency (epoch is specified to 2458206.6390453056)....
lc data points: 392  bin number: 11
Folded Period: 0.0031599 d  0.075837 hr  4.550244 min
binned_amplitude: 0.006698961634879708
epoch_time: 2458206.6390453056
>> Folded results have been saved to ./Temp/ztf_ana_test/test_dwd_zg_S04_PhaseProfile_PhaseCorrected.csv
>> Ephemeries: JD(minimum) = 2458206.6390453056 + 0.0031598917667179857 * E
>>> Plot minimum times....
pridicted min times have been saved to ./Temp/ztf_ana_test/test_dwd_zg_S04_MinTimes.csv
Peak is not significant. Analysis ended.
-------- Analyzing [zr] lc.... --------
 —> flux_std: 22.332   mean_flux_err: 21.423   ratio: 1.042
 —> Variation significant! ratio = 1.044 > 1.04
>>>> S01: Calculating LSP....
flatten lc with window_length = 501   break_tolerance =  10
>> Calculating Lomb-Scargle Periodogram....
LC data point number: 616
Flattened lc with window length: 501
min f 220.0 1 / d 	max f 360.0 1 / d
max p 0.109 h 		min p 0.067 h
samples per 

>>>> S04: Determine minimum epoch....
lc start time: 2458312.944419937
>>> fold lc with best frequency (epoch is specified to 2458312.944419937)....
lc data points: 616  bin number: 25
Folded Period: 0.0030146 d  0.072351 hr  4.341031 min
binned_amplitude: 0.008471181062041977
epoch_time: 2458312.944419937
original minimum phase: 1.0
>> corrected Epoch: 2458312.948941845


>>> fold lc with best frequency (epoch is specified to 2458312.948941845)....
lc data points: 616  bin number: 11
Folded Period: 0.0030146 d  0.072351 hr  4.341031 min
binned_amplitude: 0.005125577894439959
epoch_time: 2458312.948941845
>> Folded results have been saved to ./Temp/ztf_ana_test/test_dwd_zr_S04_PhaseProfile_PhaseCorrected.csv
>> Ephemeries: JD(minimum) = 2458312.948941845 + 0.003014605196324283 * E
>>> Plot minimum times....
pridicted min times have been saved to ./Temp/ztf_ana_test/test_dwd_zr_S04_MinTimes.csv
 -> Peak is significant. Proceeding to find more peaks....
>>>> S05: Find peaks in LSP....
>>> Find all peaks....
height 0.049403550664744285    distance 15
>>> Find result:
find peaks: 0 
peak index: [] 
peak heights: [] 
peak frequencies: [] 1 / d 
peak periods: [] d

>>>> S06: Fold lc with all peaks...
>>>> Assessing periods in zg and zr band....
period_zg = 4.55 min	period_zr = 4.34 min
period difference = 4.82 %
zg_FAL_ratio: 0.887	prob-[0.005]    zr_FAL_ratio